# AOA/AOA++-Optimized BRF Pipeline for the Larger Cardiovascular Dataset
# Reproducibility notebook for the Heliyon revised manuscript

# ==============================================================================
# PART 1: SETUP, DATA LOADING, PREPROCESSING & VISUALIZATION
# ==============================================================================

In [ ]:
# ===== Block 1 — Imports & Global Settings (Fully Anchored for Reproducibility across Servers) =====

import os
import random
import time
import warnings
import psutil

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from scipy import sparse
from scipy import stats

from IPython.display import display

from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.pipeline import Pipeline as SkPipeline
from imblearn.pipeline import Pipeline as ImbPipeline

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, RobustScaler
from sklearn.impute import SimpleImputer

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_val_predict,
    cross_val_score
)

from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    recall_score,
    precision_score,
    f1_score,
    confusion_matrix,
    RocCurveDisplay,
    PrecisionRecallDisplay,
    balanced_accuracy_score
)

from sklearn.calibration import CalibratedClassifierCV
from sklearn.inspection import permutation_importance
from sklearn.feature_selection import mutual_info_classif

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

from imblearn.ensemble import BalancedRandomForestClassifier
from imblearn.combine import SMOTEENN

try:
    from xgboost import XGBClassifier
except Exception:
    XGBClassifier = None


# -----------------------------------------------------------------------------------------
# Reproducibility
# -----------------------------------------------------------------------------------------
SEED = 42

def set_all_seeds(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

set_all_seeds(SEED)


# -----------------------------------------------------------------------------------------
# CPU 80% control
# -----------------------------------------------------------------------------------------
physical_cores = psutil.cpu_count(logical=False) or psutil.cpu_count(logical=True) or 1
allowed_cores = max(1, int(physical_cores * 0.8))

os.environ["OMP_NUM_THREADS"] = str(allowed_cores)
os.environ["MKL_NUM_THREADS"] = str(allowed_cores)
os.environ["OPENBLAS_NUM_THREADS"] = str(allowed_cores)
os.environ["NUMEXPR_NUM_THREADS"] = str(allowed_cores)

try:
    proc = psutil.Process(os.getpid())
    cpus = proc.cpu_affinity()
    keep_n = max(1, int(len(cpus) * 0.8))
    proc.cpu_affinity(cpus[:keep_n])
except Exception:
    pass


# -----------------------------------------------------------------------------------------
# Temperature-aware cooling
# -----------------------------------------------------------------------------------------
TEMP_THRESHOLD_C = 82.0
TEMP_RESUME_C = 75.0
COOL_DOWN_SECONDS = 5
MAX_TEMP_WAIT_CYCLES = 30

def get_max_cpu_temp():
    try:
        temps = psutil.sensors_temperatures(fahrenheit=False)
        if not temps:
            return None

        readings = []
        for _, entries in temps.items():
            for e in entries:
                if e.current is not None:
                    readings.append(float(e.current))

        return max(readings) if readings else None
    except Exception:
        return None


def cooling_pause(stage="", force_short_pause=False):
    temp = get_max_cpu_temp()

    if temp is None:
        if force_short_pause and COOL_DOWN_SECONDS > 0:
            time.sleep(COOL_DOWN_SECONDS)
        return

    if temp >= TEMP_THRESHOLD_C:
        print(
            f"🌡️ High CPU temperature detected at {temp:.1f}°C "
            f"during {stage}. Cooling down..."
        )

        wait_cycles = 0

        while temp is not None and temp > TEMP_RESUME_C and wait_cycles < MAX_TEMP_WAIT_CYCLES:
            time.sleep(COOL_DOWN_SECONDS)
            temp = get_max_cpu_temp()
            wait_cycles += 1

        if temp is not None:
            print(f"✅ Temperature after cooling: {temp:.1f}°C")

    elif force_short_pause and COOL_DOWN_SECONDS > 0:
        time.sleep(COOL_DOWN_SECONDS)


# -----------------------------------------------------------------------------------------
# Display / style
# -----------------------------------------------------------------------------------------
sns.set_theme(style="whitegrid")
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)

print("=" * 80)
print("✅ Global settings initialized")
print(f"Allowed CPU cores: {allowed_cores}")
print("=" * 80)

In [ ]:
# ===== Block 2: Load Data & Define Columns =====

# -----------------------------------------------------------------------------------------
# 1) Load local dataset
# -----------------------------------------------------------------------------------------
local_file = "cardio_train.csv"

if not os.path.exists(local_file):
    alt_file = "/mnt/data/cardio_train.csv"
    if os.path.exists(alt_file):
        local_file = alt_file
    else:
        raise FileNotFoundError(
            "❌ cardio_train.csv not found. Put it in the notebook directory "
            "or in /mnt/data/cardio_train.csv"
        )

df_raw = pd.read_csv(local_file, sep=";")

print("=" * 80)
print("✅ Raw dataset loaded")
print("Raw shape:", df_raw.shape)
print("Raw columns:", list(df_raw.columns))
print("=" * 80)


# -----------------------------------------------------------------------------------------
# 2) Detect target column
# -----------------------------------------------------------------------------------------
target_candidates = [
    c for c in df_raw.columns
    if c.lower() in ["target", "output", "disease", "num", "cardio"]
]

if not target_candidates:
    raise ValueError("❌ Target column not found. Expected one of: target/output/disease/num/cardio")

target_col = target_candidates[0]


# -----------------------------------------------------------------------------------------
# 3) Basic copy + remove id before duplicate detection
# -----------------------------------------------------------------------------------------
df = df_raw.copy()

if "id" in df.columns:
    df = df.drop(columns=["id"])

initial_rows = len(df)

# حذف duplicate واقعی بدون id
duplicate_count = df.duplicated().sum()
df = df.drop_duplicates().reset_index(drop=True)

print(f"Removed duplicates without id: {duplicate_count}")


# -----------------------------------------------------------------------------------------
# 4) Target cleaning
# -----------------------------------------------------------------------------------------
df[target_col] = pd.to_numeric(df[target_col], errors="coerce")

df = df.dropna(subset=[target_col]).copy()

if df[target_col].nunique() > 2:
    df[target_col] = (df[target_col] > 0).astype(int)
else:
    df[target_col] = df[target_col].astype(int)


# -----------------------------------------------------------------------------------------
# 5) Feature engineering: age_years + BMI
# Cardio dataset age is in days.
# -----------------------------------------------------------------------------------------
if "age" in df.columns:
    df["age_years"] = np.floor(df["age"] / 365.25).astype(int)
    df = df.drop(columns=["age"])

if {"height", "weight"}.issubset(df.columns):
    df["bmi"] = df["weight"] / ((df["height"] / 100) ** 2)


# -----------------------------------------------------------------------------------------
# 6) Clinical plausibility filtering
# -----------------------------------------------------------------------------------------
before_filter = len(df)

filters = pd.Series(True, index=df.index)

if "height" in df.columns:
    filters &= df["height"].between(120, 220)

if "weight" in df.columns:
    filters &= df["weight"].between(35, 250)

if "ap_hi" in df.columns:
    filters &= df["ap_hi"].between(70, 250)

if "ap_lo" in df.columns:
    filters &= df["ap_lo"].between(40, 150)

if {"ap_hi", "ap_lo"}.issubset(df.columns):
    filters &= df["ap_hi"] >= df["ap_lo"]

if "age_years" in df.columns:
    filters &= df["age_years"].between(18, 100)

if "bmi" in df.columns:
    filters &= df["bmi"].between(10, 80)

df = df.loc[filters].reset_index(drop=True)

removed_outliers = before_filter - len(df)

print(f"Removed clinically implausible rows: {removed_outliers}")


# -----------------------------------------------------------------------------------------
# 7) Define X and y
# -----------------------------------------------------------------------------------------
y = df[target_col].astype(int).copy()
X = df.drop(columns=[target_col]).copy()


# -----------------------------------------------------------------------------------------
# 8) Define categorical and numeric columns
# Cardio categorical variables:
# gender, cholesterol, gluc, smoke, alco, active
# -----------------------------------------------------------------------------------------
likely_categorical = {
    "gender",
    "cholesterol",
    "gluc",
    "smoke",
    "alco",
    "active"
}

cat_cols = [
    c for c in X.columns
    if c.lower() in likely_categorical or X[c].dtype == "object" or str(X[c].dtype) == "category"
]

num_cols = [
    c for c in X.columns
    if c not in cat_cols
]


# -----------------------------------------------------------------------------------------
# 9) Final data report
# -----------------------------------------------------------------------------------------
print("\n" + "=" * 80)
print("✅ Cleaned dataset prepared")
print(f"Initial rows without id: {initial_rows}")
print(f"Final rows after cleaning: {len(df)}")
print(f"Total removed rows: {initial_rows - len(df)}")
print("X shape:", X.shape)
print("Target distribution count:", y.value_counts().to_dict())
print("Target distribution ratio:", y.value_counts(normalize=True).round(4).to_dict())
print("Numeric columns:", num_cols)
print("Categorical columns:", cat_cols)
print("=" * 80)

display(df.head())

In [ ]:
# ===== Block 3: Exploratory Data Analysis (EDA) =====

print("=" * 80)
print("📊 Exploratory Data Analysis on CLEANED Cardio Dataset")
print("=" * 80)

print("Data shape:", X.shape)
print("Target distribution:")
display(
    pd.DataFrame({
        "count": y.value_counts(),
        "ratio": y.value_counts(normalize=True).round(4)
    })
)

print("\nNumeric columns:", num_cols)
print("Categorical columns:", cat_cols)


# -----------------------------------------------------------------------------------------
# 1) Target distribution
# -----------------------------------------------------------------------------------------
plt.figure(figsize=(5, 4))
sns.countplot(x=y, palette="Set2")
plt.title("Target Distribution: Cardiovascular Disease")
plt.xlabel("cardio: 0 = No disease, 1 = Disease")
plt.ylabel("Count")
plt.tight_layout()
plt.show()


# -----------------------------------------------------------------------------------------
# 2) Numeric summaries
# -----------------------------------------------------------------------------------------
print("\n--- Numeric Summary ---")
display(X[num_cols].describe().T)


# -----------------------------------------------------------------------------------------
# 3) Histograms for numeric features
# -----------------------------------------------------------------------------------------
if len(num_cols) > 0:
    X[num_cols].hist(
        bins=30,
        figsize=(14, 9),
        color="skyblue",
        edgecolor="black"
    )
    plt.suptitle("Histograms of Numeric Features After Cleaning", y=1.02)
    plt.tight_layout()
    plt.show()


# -----------------------------------------------------------------------------------------
# 4) Boxplots for numeric features
# -----------------------------------------------------------------------------------------
if len(num_cols) > 0:
    plt.figure(figsize=(12, 6))
    sns.boxplot(data=X[num_cols], palette="Set3")
    plt.title("Boxplots of Numeric Features After Cleaning")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()


# -----------------------------------------------------------------------------------------
# 5) Correlation heatmap
# -----------------------------------------------------------------------------------------
if len(num_cols) > 1:
    plt.figure(figsize=(9, 7))
    sns.heatmap(
        X[num_cols].corr(),
        annot=True,
        cmap="coolwarm",
        fmt=".2f",
        square=True
    )
    plt.title("Correlation Heatmap: Numeric Features")
    plt.tight_layout()
    plt.show()


# -----------------------------------------------------------------------------------------
# 6) Categorical features vs target
# -----------------------------------------------------------------------------------------
for c in cat_cols:
    plt.figure(figsize=(6, 4))
    sns.countplot(
        data=pd.concat([X[[c]], y.rename("cardio")], axis=1),
        x=c,
        hue="cardio",
        palette="Set1"
    )
    plt.title(f"{c} vs Cardiovascular Disease")
    plt.tight_layout()
    plt.show()


# -----------------------------------------------------------------------------------------
# 7) Mutual information for numeric features only
# EDA only. This is not used for model selection, so it will not leak into final test.
# -----------------------------------------------------------------------------------------
if len(num_cols) > 0:
    imputer = SimpleImputer(strategy="median")
    X_num_imp = imputer.fit_transform(X[num_cols])

    mi = mutual_info_classif(
        X_num_imp,
        y,
        random_state=SEED
    )

    mi_series = pd.Series(
        mi,
        index=num_cols
    ).sort_values(ascending=False)

    print("\n--- Mutual Information: Numeric Features ---")
    display(mi_series.to_frame("MI Score"))

    plt.figure(figsize=(8, 6))
    sns.barplot(
        x=mi_series.values,
        y=mi_series.index,
        color="salmon"
    )
    plt.title("Numeric Features by Mutual Information")
    plt.xlabel("MI Score")
    plt.tight_layout()
    plt.show()


# -----------------------------------------------------------------------------------------
# 8) Simple EDA-only baseline on numeric features
# This is diagnostic only, not the final model.
# -----------------------------------------------------------------------------------------
if len(num_cols) > 0:
    cv_eda = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=SEED
    )

    lr = LogisticRegression(
        max_iter=500,
        class_weight="balanced",
        random_state=SEED
    )

    X_num_imp = SimpleImputer(strategy="median").fit_transform(X[num_cols])

    oof = cross_val_predict(
        lr,
        X_num_imp,
        y,
        cv=cv_eda,
        method="predict_proba",
        n_jobs=1
    )[:, 1]

    print(
        "\nEDA-only Logistic Regression ROC-AUC:",
        round(roc_auc_score(y, oof), 4)
    )


# -----------------------------------------------------------------------------------------
# 9) Normality check using Shapiro on sample
# -----------------------------------------------------------------------------------------
print("\n--- Shapiro Normality Test on Numeric Features sample ---")
for c in num_cols:
    sample_size = min(len(X), 5000)
    sample = X[c].dropna().sample(
        n=sample_size,
        random_state=SEED
    )

    stat, p = stats.shapiro(sample)

    print(
        f"{c:<15} | p={p:.4f} | "
        f"{'Not Normal' if p < 0.05 else 'Normal'}"
    )

print("\n✅ EDA completed on cleaned dataset.")

In [ ]:
# ===== Block 4: Pipeline & Utils ===


# -----------------------------------------------------------------------------------------
# 1) Preprocessing builder
# -----------------------------------------------------------------------------------------
def make_preprocessor(num_cols, cat_cols):
    num_pipe = SkPipeline([
        ("imp", SimpleImputer(strategy="median")),
        ("scaler", RobustScaler())
    ])

    cat_pipe = SkPipeline([
        ("imp", SimpleImputer(strategy="most_frequent")),
        ("ohe", OneHotEncoder(handle_unknown="ignore"))
    ])

    preprocessor = ColumnTransformer([
        ("num", num_pipe, num_cols),
        ("cat", cat_pipe, cat_cols)
    ])

    return preprocessor


# -----------------------------------------------------------------------------------------
# 2) General pipeline builder
# sampler must be inside ImbPipeline to avoid SMOTEENN leakage.
# -----------------------------------------------------------------------------------------
def build_pipeline(model, num_cols, cat_cols, sampler=None):
    pre = make_preprocessor(num_cols, cat_cols)

    steps = [("pre", pre)]

    if sampler is not None:
        steps.append(("sampler", sampler))

    steps.append(("model", model))

    return ImbPipeline(steps)


# -----------------------------------------------------------------------------------------
# 3) CV evaluation utility
# Important:
# n_jobs=1 here avoids nested parallelism.
# Estimators themselves should use n_jobs=allowed_cores.
# -----------------------------------------------------------------------------------------
def evaluate_pipeline(pipe, X, y, cv, threshold=0.5, plot=True, title_prefix="OOF"):
    proba = cross_val_predict(
        pipe,
        X,
        y,
        cv=cv,
        method="predict_proba",
        n_jobs=1
    )[:, 1]

    preds = (proba >= threshold).astype(int)

    metrics = {
        "accuracy": accuracy_score(y, preds),
        "roc_auc": roc_auc_score(y, proba),
        "recall": recall_score(y, preds, zero_division=0),
        "precision": precision_score(y, preds, zero_division=0),
        "f1": f1_score(y, preds, zero_division=0),
        "balanced_accuracy": balanced_accuracy_score(y, preds)
    }

    if plot:
        plt.figure(figsize=(6, 5))
        RocCurveDisplay.from_predictions(
            y,
            proba,
            ax=plt.gca()
        )
        plt.title(f"ROC Curve ({title_prefix})")
        plt.tight_layout()
        plt.show()

        plt.figure(figsize=(6, 5))
        PrecisionRecallDisplay.from_predictions(
            y,
            proba,
            ax=plt.gca()
        )
        plt.title(f"Precision-Recall Curve ({title_prefix})")
        plt.tight_layout()
        plt.show()

    return proba, metrics


# -----------------------------------------------------------------------------------------
# 4) Threshold optimization utility
# Use only training / OOF predictions, never test predictions.
# -----------------------------------------------------------------------------------------
def find_best_threshold(y_true, proba, metric="f1"):
    thresholds = np.linspace(0.01, 0.99, 199)

    best_t = 0.5
    best_val = -np.inf

    for t in thresholds:
        preds = (proba >= t).astype(int)

        if metric == "f1":
            val = f1_score(y_true, preds, zero_division=0)

        elif metric == "recall":
            val = recall_score(y_true, preds, zero_division=0)

        elif metric == "precision":
            val = precision_score(y_true, preds, zero_division=0)

        elif metric == "balanced_acc":
            val = balanced_accuracy_score(y_true, preds)

        else:
            val = f1_score(y_true, preds, zero_division=0)

        if val > best_val:
            best_val = val
            best_t = t

    return best_t, best_val


# -----------------------------------------------------------------------------------------
# 5) Calibrated classifier compatibility helper
# -----------------------------------------------------------------------------------------
def make_calibrated_classifier(base_estimator, method="isotonic", cv=5):
    try:
        return CalibratedClassifierCV(
            estimator=base_estimator,
            method=method,
            cv=cv
        )
    except TypeError:
        return CalibratedClassifierCV(
            base_estimator=base_estimator,
            method=method,
            cv=cv
        )


# -----------------------------------------------------------------------------------------
# 6) Helper to get feature names after preprocessing
# Useful for SHAP / feature importance later.
# -----------------------------------------------------------------------------------------
def get_feature_names_from_preprocessor(fitted_preprocessor):
    try:
        return list(fitted_preprocessor.get_feature_names_out())
    except Exception:
        return None


# -----------------------------------------------------------------------------------------
# 7) Standard CV object
# -----------------------------------------------------------------------------------------
cv5 = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=SEED
)

cv10 = StratifiedKFold(
    n_splits=10,
    shuffle=True,
    random_state=SEED
)

print("✅ Pipeline and utility functions are ready.")
print("Reminder: use n_jobs=allowed_cores inside RF/BRF/XGB estimators, and n_jobs=1 at CV level.")

# ==============================================================================
# PART 2: FINAL EVALUATION, BASELINES & XAI
# ==============================================================================

In [ ]:
# ===== Block 5 Standard Baselines 
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_val_predict,
    cross_val_score,
    GridSearchCV
)

from sklearn.metrics import (
    roc_auc_score,
    f1_score,
    recall_score,
    precision_score,
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    RocCurveDisplay,
    PrecisionRecallDisplay
)

from sklearn.calibration import CalibratedClassifierCV
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

try:
    from xgboost import XGBClassifier
except Exception:
    XGBClassifier = None

try:
    from skopt import BayesSearchCV
except Exception:
    BayesSearchCV = None


print("=" * 95)
print("📊 BLOCK 5: STANDARD BASELINES — EXACT ORIGINAL MODEL SET")
print("=" * 95)

print("Baseline models:")
print("1) LR (Base)")
print("2) SVM (Base)")
print("3) XGB (Base)")
print("4) RF (Default)")
print("5) RF (GridSearch)")
print("6) RF (Bayesian Opt)")
print("=" * 95)


# -----------------------------------------------------------------------------------------
# 0) CPU / cooling compatibility
# -----------------------------------------------------------------------------------------
try:
    allowed_cores
except NameError:
    import psutil
    physical_cores = psutil.cpu_count(logical=False) or psutil.cpu_count(logical=True) or 1
    allowed_cores = max(1, int(physical_cores * 0.8))

try:
    cooling_pause
except NameError:
    def cooling_pause(stage="", force_short_pause=False):
        if force_short_pause:
            time.sleep(5)

print(f"Allowed CPU cores inside estimators: {allowed_cores}")
print("CV-level n_jobs is set to 1 to avoid nested parallel CPU overload.")


# -----------------------------------------------------------------------------------------
# 1) Leak-safe Train/Test split
# This split must be reused in later AOA / AOA++ blocks.
# -----------------------------------------------------------------------------------------
print("\n>>> Splitting data into train/test...")

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=SEED
)

print(f"Train shape: {X_train.shape}")
print(f"Test shape : {X_test.shape}")
print("Train target distribution:", y_train.value_counts().to_dict())
print("Test target distribution :", y_test.value_counts().to_dict())


# -----------------------------------------------------------------------------------------
# 2) FeatureNameSelector — fixed for raw feature names + transformed feature names
# -----------------------------------------------------------------------------------------
class FeatureNameSelector(BaseEstimator, TransformerMixin):
    def __init__(self, preprocessor=None, selected_feature_names=None):
        self.preprocessor = preprocessor
        self.selected_feature_names = selected_feature_names if selected_feature_names is not None else []

    def fit(self, X, y=None):
        self.preprocessor_ = clone(self.preprocessor)
        self.preprocessor_.fit(X, y)

        all_feats = np.asarray(self.preprocessor_.get_feature_names_out())
        selected = set(map(str, self.selected_feature_names))

        selected_indices = []

        for i, fname in enumerate(all_feats):
            fname = str(fname)
            clean = fname.split("__", 1)[-1]

            keep = False

            # numeric case: num__age_years -> age_years
            if fname in selected or clean in selected:
                keep = True

            # one-hot case: cat__cholesterol_2 -> cholesterol
            for s in selected:
                if clean == s or clean.startswith(s + "_"):
                    keep = True
                    break

            if keep:
                selected_indices.append(i)

        self.feature_names_out_ = all_feats
        self.selected_indices_ = np.array(selected_indices, dtype=int)

        if len(self.selected_indices_) == 0:
            raise ValueError(
                "No selected features matched transformed feature names. "
                "Check selected_feature_names, num_cols, and cat_cols."
            )

        return self

    def transform(self, X):
        Xt = self.preprocessor_.transform(X)
        return Xt[:, self.selected_indices_]


# -----------------------------------------------------------------------------------------
# 3) Correct Permutation Importance → Top-K features
# Important:
# permutation_importance on a pipeline with raw X returns importances for RAW X columns,
# not transformed OneHot columns.
# -----------------------------------------------------------------------------------------
def compute_perm_importance_topk(X_tr, y_tr, num_cols, cat_cols, K=15, seed=42):
    temp_model = RandomForestClassifier(
        n_estimators=300,
        class_weight="balanced",
        random_state=seed,
        n_jobs=allowed_cores
    )

    temp_pipe = build_pipeline(
        model=temp_model,
        num_cols=num_cols,
        cat_cols=cat_cols,
        sampler=None
    )

    temp_pipe.fit(X_tr, y_tr)

    result = permutation_importance(
        temp_pipe,
        X_tr,
        y_tr,
        scoring="roc_auc",
        n_repeats=10,
        random_state=seed,
        n_jobs=1
    )

    imp_df = pd.DataFrame({
        "feature": X_tr.columns,
        "mean": result.importances_mean,
        "std": result.importances_std
    }).sort_values(
        by="mean",
        ascending=False
    ).reset_index(drop=True)

    top_features = imp_df.head(K)["feature"].tolist()

    plt.figure(figsize=(8, 6))
    sns.barplot(
        data=imp_df.head(K),
        x="mean",
        y="feature",
        orient="h"
    )
    plt.title(f"Top-{K} Raw Features by Permutation Importance")
    plt.xlabel("Mean ROC-AUC decrease")
    plt.ylabel("Feature")
    plt.tight_layout()
    plt.show()

    return top_features, imp_df


# -----------------------------------------------------------------------------------------
# 4) Compute Top-15 features only on X_train
# -----------------------------------------------------------------------------------------
print("\n>>> Computing Top-15 features on X_train only...")

global_top_features, baseline_imp_df = compute_perm_importance_topk(
    X_train,
    y_train,
    num_cols,
    cat_cols,
    K=15,
    seed=SEED
)

print("Selected Top-15 features:", global_top_features)


# -----------------------------------------------------------------------------------------
# 5) Baseline calibrated pipeline helper
# Same logic as your original baseline: Feature Selection + Isotonic Calibration.
# -----------------------------------------------------------------------------------------
def make_baseline_pipeline(base_estimator, selected_feats):
    preprocessor = make_preprocessor(
        num_cols=num_cols,
        cat_cols=cat_cols
    )

    feat_sel = FeatureNameSelector(
        preprocessor=preprocessor,
        selected_feature_names=selected_feats
    )

    calibrated_model = make_calibrated_classifier(
        base_estimator,
        method="isotonic",
        cv=5
    )

    return ImbPipeline([
        ("feat_pre", feat_sel),
        ("model", calibrated_model)
    ])


# -----------------------------------------------------------------------------------------
# 6) Bootstrap CI helper
# -----------------------------------------------------------------------------------------
def compute_bootstrap_metrics_with_ci(
    y_true,
    y_prob,
    threshold,
    n_bootstraps=1000,
    seed=42
):
    rng = np.random.RandomState(seed)

    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)

    boot_aucs = []
    boot_accs = []
    boot_recalls = []
    boot_precisions = []
    boot_f1s = []

    for _ in range(n_bootstraps):
        idx = rng.randint(0, len(y_true), len(y_true))

        if len(np.unique(y_true[idx])) < 2:
            continue

        yp = y_prob[idx]
        yt = y_true[idx]
        pred = (yp >= threshold).astype(int)

        boot_aucs.append(roc_auc_score(yt, yp))
        boot_accs.append(accuracy_score(yt, pred))
        boot_recalls.append(recall_score(yt, pred, zero_division=0))
        boot_precisions.append(precision_score(yt, pred, zero_division=0))
        boot_f1s.append(f1_score(yt, pred, zero_division=0))

    return {
        "auc_ci": np.percentile(boot_aucs, [2.5, 97.5]),
        "acc_ci": np.percentile(boot_accs, [2.5, 97.5]),
        "recall_ci": np.percentile(boot_recalls, [2.5, 97.5]),
        "precision_ci": np.percentile(boot_precisions, [2.5, 97.5]),
        "f1_ci": np.percentile(boot_f1s, [2.5, 97.5]),
    }


def format_ci(value, ci):
    return f"{value:.3f} [{ci[0]:.3f}-{ci[1]:.3f}]"


# -----------------------------------------------------------------------------------------
# 7) RF GridSearch + Bayesian HPO setup
# Same baseline algorithms as yours.
# -----------------------------------------------------------------------------------------
rf_param_space_grid = {
    "model__estimator__n_estimators": [150, 300, 500],
    "model__estimator__max_depth": [5, 10, None],
    "model__estimator__min_samples_split": [2, 5, 10]
}

raw_rf_for_hpo = RandomForestClassifier(
    class_weight="balanced",
    random_state=SEED,
    n_jobs=allowed_cores
)

base_pipe_for_hpo = make_baseline_pipeline(
    raw_rf_for_hpo,
    global_top_features
)

cv_inner = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=SEED
)

print("\n>>> Preparing GridSearch and Bayesian Optimization baselines...")

grid_search_baseline = GridSearchCV(
    estimator=base_pipe_for_hpo,
    param_grid=rf_param_space_grid,
    cv=cv_inner,
    scoring="roc_auc",
    n_jobs=1,
    refit=True
)

if BayesSearchCV is not None:
    bayesian_search_baseline = BayesSearchCV(
        estimator=base_pipe_for_hpo,
        search_spaces=rf_param_space_grid,
        n_iter=10,
        cv=cv_inner,
        scoring="roc_auc",
        n_jobs=1,
        random_state=SEED,
        refit=True
    )
else:
    raise ImportError(
        "skopt is not installed. Your original baseline uses BayesSearchCV. "
        "Install it with: pip install scikit-optimize"
    )


# -----------------------------------------------------------------------------------------
# 8) Exact original baseline models
# -----------------------------------------------------------------------------------------
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

if XGBClassifier is None:
    raise ImportError(
        "xgboost is not installed. Your original baseline includes XGB (Base). "
        "Install it with: pip install xgboost"
    )

baselines = {
    "LR (Base)": LogisticRegression(
        class_weight="balanced",
        max_iter=3000,
        solver="liblinear",
        random_state=SEED
    ),

    "SVM (Base)": SVC(
        class_weight="balanced",
        kernel="rbf",
        probability=True,
        random_state=SEED
    ),

    "XGB (Base)": XGBClassifier(
        scale_pos_weight=scale_pos_weight,
        eval_metric="logloss",
        n_estimators=200,
        random_state=SEED,
        n_jobs=allowed_cores
    ),

    "RF (Default)": RandomForestClassifier(
        class_weight="balanced",
        n_estimators=100,
        random_state=SEED,
        n_jobs=allowed_cores
    ),

    "RF (GridSearch)": grid_search_baseline,

    "RF (Bayesian Opt)": bayesian_search_baseline
}


# -----------------------------------------------------------------------------------------
# 9) Evaluation
# -----------------------------------------------------------------------------------------
cv_outer = StratifiedKFold(
    n_splits=10,
    shuffle=True,
    random_state=SEED
)

baseline_results = []
baseline_model_folds = {}
baseline_fitted_pipelines = {}

print("\n>>> Running exact baseline models...")

for name, model in baselines.items():
    print("\n" + "=" * 95)
    print(f"🚀 Baseline model: {name}")
    print("=" * 95)

    cooling_pause(
        stage=f"before baseline {name}",
        force_short_pause=True
    )

    if isinstance(model, (GridSearchCV, BayesSearchCV)):
        pipe = model
    else:
        pipe = make_baseline_pipeline(
            model,
            global_top_features
        )

    # OOF train probability
    proba_train = cross_val_predict(
        pipe,
        X_train,
        y_train,
        cv=cv_outer,
        method="predict_proba",
        n_jobs=1
    )[:, 1]

    pred_train_05 = (proba_train >= 0.5).astype(int)

    t_opt, _ = find_best_threshold(
        y_train,
        proba_train,
        metric="f1"
    )

    pred_train_opt = (proba_train >= t_opt).astype(int)

    train_auc = roc_auc_score(y_train, proba_train)
    train_acc = accuracy_score(y_train, pred_train_05)
    train_rec = recall_score(y_train, pred_train_05, zero_division=0)
    train_pre = precision_score(y_train, pred_train_05, zero_division=0)
    train_f1 = f1_score(y_train, pred_train_05, zero_division=0)

    train_acc_opt = accuracy_score(y_train, pred_train_opt)
    train_rec_opt = recall_score(y_train, pred_train_opt, zero_division=0)
    train_pre_opt = precision_score(y_train, pred_train_opt, zero_division=0)
    train_f1_opt = f1_score(y_train, pred_train_opt, zero_division=0)

    # Fit final model on full training set
    pipe.fit(X_train, y_train)

    baseline_fitted_pipelines[name] = pipe

    cooling_pause(
        stage=f"after fit baseline {name}",
        force_short_pause=True
    )

    # Test probability
    proba_test = pipe.predict_proba(X_test)[:, 1]

    pred_test_05 = (proba_test >= 0.5).astype(int)
    pred_test_opt = (proba_test >= t_opt).astype(int)

    test_auc = roc_auc_score(y_test, proba_test)
    test_acc = accuracy_score(y_test, pred_test_05)
    test_rec = recall_score(y_test, pred_test_05, zero_division=0)
    test_pre = precision_score(y_test, pred_test_05, zero_division=0)
    test_f1 = f1_score(y_test, pred_test_05, zero_division=0)

    test_acc_opt = accuracy_score(y_test, pred_test_opt)
    test_rec_opt = recall_score(y_test, pred_test_opt, zero_division=0)
    test_pre_opt = precision_score(y_test, pred_test_opt, zero_division=0)
    test_f1_opt = f1_score(y_test, pred_test_opt, zero_division=0)

    ci = compute_bootstrap_metrics_with_ci(
        y_test,
        proba_test,
        threshold=0.5,
        n_bootstraps=1000,
        seed=SEED
    )

    # Fold scores for statistical analysis
    fold_scores = cross_val_score(
        pipe,
        X_train,
        y_train,
        cv=cv_outer,
        scoring="roc_auc",
        n_jobs=1
    )

    baseline_model_folds[name] = fold_scores

    print(f"\n=== {name} ===")
    print("Train (OOF) @0.50:", {
        "accuracy": round(train_acc, 4),
        "roc_auc": round(train_auc, 4),
        "recall": round(train_rec, 4),
        "precision": round(train_pre, 4),
        "f1": round(train_f1, 4)
    })

    print("Train (OOF) @opt :", {
        "accuracy": round(train_acc_opt, 4),
        "roc_auc": round(train_auc, 4),
        "recall": round(train_rec_opt, 4),
        "precision": round(train_pre_opt, 4),
        "f1": round(train_f1_opt, 4),
        "threshold": round(t_opt, 4)
    })

    print("Test        @0.50:", {
        "accuracy": round(test_acc, 4),
        "roc_auc": round(test_auc, 4),
        "recall": round(test_rec, 4),
        "precision": round(test_pre, 4),
        "f1": round(test_f1, 4)
    })

    print("Test        @opt :", {
        "accuracy": round(test_acc_opt, 4),
        "roc_auc": round(test_auc, 4),
        "recall": round(test_rec_opt, 4),
        "precision": round(test_pre_opt, 4),
        "f1": round(test_f1_opt, 4),
        "threshold": round(t_opt, 4)
    })

    baseline_results.append({
        "Model": name,

        "Train_AUC": train_auc,
        "Train_Accuracy_05": train_acc,
        "Train_Recall_05": train_rec,
        "Train_Precision_05": train_pre,
        "Train_F1_05": train_f1,

        "Train_Accuracy_opt": train_acc_opt,
        "Train_Recall_opt": train_rec_opt,
        "Train_Precision_opt": train_pre_opt,
        "Train_F1_opt": train_f1_opt,

        "Test_AUC": test_auc,
        "Test_AUC_95CI": format_ci(test_auc, ci["auc_ci"]),
        "Test_Accuracy_05": test_acc,
        "Test_Accuracy_95CI": format_ci(test_acc, ci["acc_ci"]),
        "Test_Recall_05": test_rec,
        "Test_Recall_95CI": format_ci(test_rec, ci["recall_ci"]),
        "Test_Precision_05": test_pre,
        "Test_Precision_95CI": format_ci(test_pre, ci["precision_ci"]),
        "Test_F1_05": test_f1,
        "Test_F1_95CI": format_ci(test_f1, ci["f1_ci"]),

        "Test_Accuracy_opt": test_acc_opt,
        "Test_Recall_opt": test_rec_opt,
        "Test_Precision_opt": test_pre_opt,
        "Test_F1_opt": test_f1_opt,

        "Optimal_Threshold": t_opt,
        "CV_ROC_AUC_Mean": np.mean(fold_scores),
        "CV_ROC_AUC_Std": np.std(fold_scores)
    })

    # ROC & PR plots
    plt.figure(figsize=(14, 6))

    plt.subplot(1, 2, 1)
    RocCurveDisplay.from_predictions(
        y_test,
        proba_test,
        ax=plt.gca()
    )
    plt.title(f"ROC - Test [{name}]")

    plt.subplot(1, 2, 2)
    PrecisionRecallDisplay.from_predictions(
        y_test,
        proba_test,
        ax=plt.gca()
    )
    plt.title(f"PR - Test [{name}]")

    plt.tight_layout()
    plt.show()

    cooling_pause(
        stage=f"between baseline models after {name}",
        force_short_pause=True
    )


# -----------------------------------------------------------------------------------------
# 10) Baseline summary table
# -----------------------------------------------------------------------------------------
baseline_results_df = pd.DataFrame(baseline_results)

baseline_results_df = baseline_results_df.sort_values(
    by="Test_AUC",
    ascending=False
).reset_index(drop=True)

print("\n" + "=" * 95)
print("📈 EXACT BASELINE RESULTS — SORTED BY TEST ROC-AUC")
print("=" * 95)

display(
    baseline_results_df[
        [
            "Model",
            "Test_AUC_95CI",
            "Test_Recall_95CI",
            "Test_F1_95CI",
            "Test_Precision_95CI",
            "Test_Accuracy_95CI",
            "Optimal_Threshold",
            "CV_ROC_AUC_Mean",
            "CV_ROC_AUC_Std"
        ]
    ]
)

baseline_results_df.to_csv(
    "baseline_exact_original_models_results.csv",
    index=False
)

pd.DataFrame(baseline_model_folds).to_csv(
    "baseline_exact_original_models_10fold_auc.csv",
    index=False
)

print("✅ Exact original baseline models completed.")
print("• Saved: baseline_exact_original_models_results.csv")
print("• Saved: baseline_exact_original_models_10fold_auc.csv")

In [ ]:
# ===== Block 6: Pairwise Statistical Comparison of Baseline Models =====

import scipy.stats as stats
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import itertools

print("=" * 95)
print("STEP 4: PAIRWISE STATISTICAL SIGNIFICANCE ANALYSIS (BASELINE MODELS ONLY)")
print("=" * 95)

# -----------------------------------------------------------------------------------------
# 0) Safety check
# -----------------------------------------------------------------------------------------
if "baseline_model_folds" not in globals():
    raise NameError(
        "❌ baseline_model_folds not found. "
        "Run the exact baseline block first."
    )

if not isinstance(baseline_model_folds, dict) or len(baseline_model_folds) == 0:
    raise ValueError(
        "❌ baseline_model_folds is empty or invalid. "
        "Check the baseline block execution."
    )

# -----------------------------------------------------------------------------------------
# 1) Extract 10-fold ROC-AUC scores for exact baseline models
# -----------------------------------------------------------------------------------------
baseline_names = list(baseline_model_folds.keys())

all_baseline_folds = {
    name: np.asarray(baseline_model_folds[name], dtype=float)
    for name in baseline_names
}

print("Baseline models included:")
for name in baseline_names:
    print(f"• {name}: {len(all_baseline_folds[name])} folds")

# Check all models have same number of folds
fold_lengths = {name: len(scores) for name, scores in all_baseline_folds.items()}

if len(set(fold_lengths.values())) != 1:
    raise ValueError(
        f"❌ Fold count mismatch across baseline models: {fold_lengths}"
    )

# -----------------------------------------------------------------------------------------
# 2) Summary table: mean/std/min/max ROC-AUC
# -----------------------------------------------------------------------------------------
baseline_stat_summary = pd.DataFrame([
    {
        "Model": name,
        "Mean_ROC_AUC": np.mean(scores),
        "Std_ROC_AUC": np.std(scores),
        "Min_ROC_AUC": np.min(scores),
        "Max_ROC_AUC": np.max(scores),
        "Fold_Scores": np.round(scores, 5)
    }
    for name, scores in all_baseline_folds.items()
]).sort_values(
    by="Mean_ROC_AUC",
    ascending=False
).reset_index(drop=True)

print("\n--- Baseline 10-Fold ROC-AUC Summary ---")
display(baseline_stat_summary)

# -----------------------------------------------------------------------------------------
# 3) Pairwise paired t-test between all baseline model pairs
# -----------------------------------------------------------------------------------------
pairwise_results = []

print("\n--- Pairwise Paired T-Test Results ---")
print(f"{'Model Pair':<60} | {'T-Stat':<10} | {'P-Value':<10} | {'Significant'}")
print("-" * 95)

for name1, name2 in itertools.combinations(baseline_names, 2):
    scores1 = all_baseline_folds[name1]
    scores2 = all_baseline_folds[name2]

    t_stat, p_val = stats.ttest_rel(scores1, scores2)

    is_sig = "✅ YES" if p_val < 0.05 else "⚠️ NO"

    print(
        f"{name1 + ' vs ' + name2:<60} | "
        f"{t_stat:>10.4f} | "
        f"{p_val:>10.4f} | "
        f"{is_sig}"
    )

    pairwise_results.append({
        "Model_1": name1,
        "Model_2": name2,
        "Mean_1": np.mean(scores1),
        "Mean_2": np.mean(scores2),
        "Mean_Diff": np.mean(scores1) - np.mean(scores2),
        "T_Stat": t_stat,
        "P_Value": p_val,
        "Significant_0.05": p_val < 0.05
    })

pairwise_results_df = pd.DataFrame(pairwise_results)

# -----------------------------------------------------------------------------------------
# 4) Stability report
# -----------------------------------------------------------------------------------------
print("\n" + "-" * 95)
print(f"{'Baseline Model':<35} | {'Mean ROC-AUC':<12} | {'Std Dev':<10} | {'Stability Rank'}")
print("-" * 95)

stability_df = baseline_stat_summary.copy()
stability_df = stability_df.sort_values(
    by="Std_ROC_AUC",
    ascending=True
).reset_index(drop=True)

stability_df["Stability_Rank"] = np.arange(1, len(stability_df) + 1)

for _, row in stability_df.iterrows():
    print(
        f"{row['Model']:<35} | "
        f"{row['Mean_ROC_AUC']:.5f}      | "
        f"{row['Std_ROC_AUC']:.5f}   | "
        f"{int(row['Stability_Rank'])}"
    )

print("=" * 95 + "\n")

# -----------------------------------------------------------------------------------------
# 5) Boxplot: baseline stability comparison
# -----------------------------------------------------------------------------------------
plt.figure(figsize=(12, 6))

plot_data = [
    all_baseline_folds[name]
    for name in baseline_names
]

try:
    box = plt.boxplot(
        plot_data,
        patch_artist=True,
        tick_labels=baseline_names,
        widths=0.5
    )
except TypeError:
    box = plt.boxplot(
        plot_data,
        patch_artist=True,
        labels=baseline_names,
        widths=0.5
    )

colors = plt.cm.plasma(
    np.linspace(0, 0.8, len(baseline_names))
)

for patch, color in zip(box["boxes"], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
    patch.set_edgecolor("black")

for median in box["medians"]:
    median.set_color("#FFB703")
    median.set_linewidth(2)

for whisk in box["whiskers"]:
    whisk.set_color("black")
    whisk.set_linewidth(1.2)

for cap in box["caps"]:
    cap.set_color("black")
    cap.set_linewidth(1.2)

plt.title(
    "Performance Stability & Statistical Distribution of Baseline Models",
    fontsize=13,
    fontweight="bold",
    pad=15
)

plt.ylabel(
    "Validation ROC-AUC Score (10-Fold CV)",
    fontsize=11,
    fontweight="bold"
)

plt.grid(
    True,
    linestyle="--",
    alpha=0.3,
    axis="y"
)

plt.xticks(
    rotation=45,
    ha="right"
)

plt.tight_layout()

plt.savefig(
    "baseline_statistical_comparison.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# -----------------------------------------------------------------------------------------
# 6) Save outputs for paper/report
# -----------------------------------------------------------------------------------------
baseline_stat_summary.to_csv(
    "baseline_statistical_summary.csv",
    index=False
)

pairwise_results_df.to_csv(
    "baseline_pairwise_ttest_results.csv",
    index=False
)

stability_df.to_csv(
    "baseline_stability_ranking.csv",
    index=False
)

print("✅ Baseline pairwise statistical analysis completed successfully.")
print("• Saved: baseline_statistical_comparison.png")
print("• Saved: baseline_statistical_summary.csv")
print("• Saved: baseline_pairwise_ttest_results.csv")
print("• Saved: baseline_stability_ranking.csv")

In [ ]:
# ===== Block 7: Full Pipelines — PI-based FS + TRUE AOA + Calibration + Evaluation =====

import os
import time
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_predict, cross_val_score
from sklearn.metrics import (
    roc_auc_score,
    accuracy_score,
    recall_score,
    precision_score,
    f1_score,
    confusion_matrix,
    RocCurveDisplay,
    PrecisionRecallDisplay
)
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline as SkPipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler, OneHotEncoder

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.combine import SMOTEENN
from imblearn.ensemble import BalancedRandomForestClassifier


print("=" * 95)
print("🚀 BLOCK 20: TRUE AOA OPTIMIZED MODELS — CARDIO PROJECT")
print("=" * 95)


# -----------------------------------------------------------------------------------------
# 0) Safety checks
# -----------------------------------------------------------------------------------------
required_vars = [
    "X_train", "X_test", "y_train", "y_test",
    "num_cols", "cat_cols",
    "allowed_cores"
]

missing_vars = [v for v in required_vars if v not in globals()]

if missing_vars:
    raise NameError(
        f"❌ Missing variables from previous blocks: {missing_vars}. "
        "Run the preprocessing and exact baseline blocks first."
    )

try:
    SEED
except NameError:
    SEED = 42

random.seed(SEED)
np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

try:
    cooling_pause
except NameError:
    def cooling_pause(stage="", force_short_pause=False):
        if force_short_pause:
            time.sleep(5)

print(f"Allowed CPU cores inside estimators: {allowed_cores}")
print("CV-level n_jobs is set to 1 to avoid nested parallel overload.")


# -----------------------------------------------------------------------------------------
# 1) Use the same selected features as baseline / AOA++
# -----------------------------------------------------------------------------------------
if "global_top_features" in globals():
    aoa_top_features = global_top_features
    print("✅ Using global_top_features from exact baseline block.")

elif "top_features" in globals():
    aoa_top_features = top_features
    print("✅ Using top_features from previous optimization block.")

else:
    print("⚠️ No saved top features found. Computing Top-15 features on X_train only...")

    if "compute_perm_importance_topk" not in globals():
        raise NameError(
            "❌ compute_perm_importance_topk not found. "
            "Run the feature-selection/baseline block first."
        )

    aoa_top_features, aoa_imp_df = compute_perm_importance_topk(
        X_train,
        y_train,
        num_cols,
        cat_cols,
        K=15,
        seed=SEED
    )

print("AOA selected features:", aoa_top_features)


# -----------------------------------------------------------------------------------------
# 2) Preprocessing and feature selector
# -----------------------------------------------------------------------------------------
def make_aoa_preprocessor(num_cols, cat_cols):
    num_pipe = SkPipeline([
        ("imp", SimpleImputer(strategy="median")),
        ("scaler", RobustScaler())
    ])

    cat_pipe = SkPipeline([
        ("imp", SimpleImputer(strategy="most_frequent")),
        ("ohe", OneHotEncoder(handle_unknown="ignore"))
    ])

    preprocessor = ColumnTransformer([
        ("num", num_pipe, num_cols),
        ("cat", cat_pipe, cat_cols)
    ])

    return preprocessor


class AOAFeatureNameSelector(BaseEstimator, TransformerMixin):
    def __init__(self, preprocessor=None, selected_feature_names=None):
        self.preprocessor = preprocessor
        self.selected_feature_names = selected_feature_names if selected_feature_names is not None else []

    def fit(self, X, y=None):
        self.preprocessor_ = clone(self.preprocessor)
        self.preprocessor_.fit(X, y)

        all_feats = np.asarray(self.preprocessor_.get_feature_names_out())
        selected = set(map(str, self.selected_feature_names))

        selected_indices = []

        for i, fname in enumerate(all_feats):
            fname = str(fname)
            clean = fname.split("__", 1)[-1]

            keep = False

            # Numeric case:
            # num__age_years -> age_years
            if fname in selected or clean in selected:
                keep = True

            # One-hot categorical case:
            # cat__cholesterol_2 -> cholesterol
            for s in selected:
                if clean == s or clean.startswith(s + "_"):
                    keep = True
                    break

            if keep:
                selected_indices.append(i)

        self.feature_names_out_ = all_feats
        self.selected_indices_ = np.array(selected_indices, dtype=int)

        if len(self.selected_indices_) == 0:
            raise ValueError(
                "❌ No selected features matched transformed feature names. "
                "Check aoa_top_features, num_cols, and cat_cols."
            )

        return self

    def transform(self, X):
        Xt = self.preprocessor_.transform(X)
        return Xt[:, self.selected_indices_]


def make_aoa_calibrated_classifier(base_estimator):
    try:
        return CalibratedClassifierCV(
            estimator=base_estimator,
            method="isotonic",
            cv=5
        )
    except TypeError:
        return CalibratedClassifierCV(
            base_estimator=base_estimator,
            method="isotonic",
            cv=5
        )


def make_aoa_fs_calibrated_pipeline(base_estimator, selected_feature_names, sampler=None):
    preprocessor = make_aoa_preprocessor(
        num_cols=num_cols,
        cat_cols=cat_cols
    )

    feat_sel = AOAFeatureNameSelector(
        preprocessor=preprocessor,
        selected_feature_names=selected_feature_names
    )

    calibrated_model = make_aoa_calibrated_classifier(base_estimator)

    steps = [("feat_pre", feat_sel)]

    if sampler is not None:
        steps.append(("sampler", sampler))

    steps.append(("model", calibrated_model))

    return ImbPipeline(steps)


# -----------------------------------------------------------------------------------------
# 3) AOA search space
# Xi is optimized in normalized continuous space [0, 1]^5.
# Then Xi is decoded to actual RF/BRF hyperparameters.
# -----------------------------------------------------------------------------------------
AOA_MAX_DEPTH_CHOICES = [None, 3, 5, 7, 10]
AOA_MAX_FEATURES_CHOICES = ["sqrt", "log2", None]


def _aoa_decode_int(z, low, high):
    z = float(np.clip(z, 0, 1))
    return int(round(low + z * (high - low)))


def _aoa_decode_choice(z, choices):
    z = float(np.clip(z, 0, 1))
    idx = int(round(z * (len(choices) - 1)))
    idx = int(np.clip(idx, 0, len(choices) - 1))
    return choices[idx]


def decode_aoa_candidate_vector(x_vec):
    x_vec = np.clip(np.asarray(x_vec, dtype=float), 0, 1)

    return {
        "n_estimators": _aoa_decode_int(x_vec[0], 150, 600),
        "max_depth": _aoa_decode_choice(x_vec[1], AOA_MAX_DEPTH_CHOICES),
        "min_samples_split": _aoa_decode_int(x_vec[2], 2, 12),
        "min_samples_leaf": _aoa_decode_int(x_vec[3], 1, 6),
        "max_features": _aoa_decode_choice(x_vec[4], AOA_MAX_FEATURES_CHOICES),
    }


# -----------------------------------------------------------------------------------------
# 4) Estimator builders
# -----------------------------------------------------------------------------------------
def build_aoa_brf(params):
    return BalancedRandomForestClassifier(
        n_estimators=params["n_estimators"],
        max_depth=params["max_depth"],
        min_samples_split=params["min_samples_split"],
        min_samples_leaf=params["min_samples_leaf"],
        max_features=params["max_features"],
        random_state=SEED,
        n_jobs=allowed_cores
    )


def build_aoa_rf(params):
    return RandomForestClassifier(
        n_estimators=params["n_estimators"],
        max_depth=params["max_depth"],
        min_samples_split=params["min_samples_split"],
        min_samples_leaf=params["min_samples_leaf"],
        max_features=params["max_features"],
        class_weight="balanced",
        random_state=SEED,
        n_jobs=allowed_cores
    )


# -----------------------------------------------------------------------------------------
# 5) TRUE AOA optimizer according to Algorithm 1
# Algorithm minimizes fitness.
# We define fitness = -weighted_score because we want to maximize score.
# AOA uses fixed alpha = 2.0.
# -----------------------------------------------------------------------------------------
def aoa_optimize_generic(
    build_estimator_fn,
    selected_feature_names,
    use_smoteenn=False,
    max_iter=15,
    pop_size=24,
    seed=42,
    label="GEN",
    alpha=2.0
):
    start_time = time.time()

    rng = np.random.RandomState(seed)
    sampler = SMOTEENN(random_state=seed) if use_smoteenn else None

    # Validation set for AOA fitness evaluation
    X_aoa_train, X_aoa_val, y_aoa_train, y_aoa_val = train_test_split(
        X_train,
        y_train,
        test_size=0.2,
        stratify=y_train,
        random_state=seed
    )

    def fitness(x_vec):
        params = decode_aoa_candidate_vector(x_vec)

        estimator = build_estimator_fn(params)

        pipe = make_aoa_fs_calibrated_pipeline(
            base_estimator=estimator,
            selected_feature_names=selected_feature_names,
            sampler=sampler
        )

        pipe.fit(X_aoa_train, y_aoa_train)

        val_proba = pipe.predict_proba(X_aoa_val)[:, 1]
        val_pred = (val_proba >= 0.5).astype(int)

        auc = roc_auc_score(y_aoa_val, val_proba)
        rec = recall_score(y_aoa_val, val_pred, zero_division=0)
        f1 = f1_score(y_aoa_val, val_pred, zero_division=0)
        acc = accuracy_score(y_aoa_val, val_pred)

        score = 0.4 * auc + 0.4 * rec + 0.1 * f1 + 0.1 * acc

        # AOA minimizes fitness
        fit_value = -score

        return fit_value, {
            "auc": auc,
            "recall": rec,
            "f1": f1,
            "acc": acc,
            "score": score,
            **params
        }

    # Initialize population Xi for i = 1 to N with random hyperparameters
    dim = 5
    population = rng.uniform(0, 1, size=(pop_size, dim))

    # Xbest ← initial best solution with lowest fitness
    best_x = None
    best_fit = np.inf
    best_mets = None
    history = []

    for i in range(pop_size):
        fit_i, mets_i = fitness(population[i])

        history.append({
            "iter": 0,
            "individual": i + 1,
            "alpha": alpha,
            "fitness": fit_i,
            **mets_i
        })

        if fit_i < best_fit:
            best_x = population[i].copy()
            best_fit = fit_i
            best_mets = mets_i.copy()

        if (i + 1) % 6 == 0:
            cooling_pause(
                stage=f"{label} initialization {i+1}/{pop_size}",
                force_short_pause=False
            )

    print(
        f"[{label}] Initial Best | "
        f"Alpha={alpha:.4f} | "
        f"Score={best_mets['score']:.4f} | "
        f"AUC={best_mets['auc']:.4f} | "
        f"F1={best_mets['f1']:.4f} | "
        f"ACC={best_mets['acc']:.4f} | "
        f"Recall={best_mets['recall']:.4f}"
    )

    # AOA main loop
    for t in range(1, max_iter + 1):

        # Compute XM ← mean of all Xi in population
        XM = np.mean(population, axis=0)

        new_population = []

        for i in range(pop_size):
            Xi = population[i].copy()

            # rand ← uniform random number in [0, 1]
            rand = rng.uniform(0, 1)

            # X'i ← Xbest + alpha * (rand - 0.5) * (XM - Xi)
            Xi_prime = best_x + alpha * (rand - 0.5) * (XM - Xi)

            # Keep Xi inside normalized bounds
            Xi_prime = np.clip(Xi_prime, 0, 1)

            # Evaluate fitness f(X'i)
            fit_prime, mets_prime = fitness(Xi_prime)

            history.append({
                "iter": t,
                "individual": i + 1,
                "alpha": alpha,
                "fitness": fit_prime,
                **mets_prime
            })

            # if f(X'i) < f(Xbest): Xbest ← X'i
            if fit_prime < best_fit:
                best_x = Xi_prime.copy()
                best_fit = fit_prime
                best_mets = mets_prime.copy()

            # Update position
            new_population.append(Xi_prime)

            if (i + 1) % 6 == 0:
                cooling_pause(
                    stage=f"{label} iter {t}, candidate {i+1}/{pop_size}",
                    force_short_pause=False
                )

        population = np.asarray(new_population)

        print(
            f"[{label}] Iter {t}/{max_iter} | "
            f"Alpha={alpha:.4f} | "
            f"Score={best_mets['score']:.4f} | "
            f"AUC={best_mets['auc']:.4f} | "
            f"F1={best_mets['f1']:.4f} | "
            f"ACC={best_mets['acc']:.4f} | "
            f"Recall={best_mets['recall']:.4f}"
        )

        cooling_pause(
            stage=f"{label} iteration {t}",
            force_short_pause=True
        )

    best_params = decode_aoa_candidate_vector(best_x)

    # 10-fold scores for statistical analysis
    best_pipe = make_aoa_fs_calibrated_pipeline(
        base_estimator=build_estimator_fn(best_params),
        selected_feature_names=selected_feature_names,
        sampler=sampler
    )

    cv_for_folds = StratifiedKFold(
        n_splits=10,
        shuffle=True,
        random_state=seed
    )

    best_folds = cross_val_score(
        best_pipe,
        X_train,
        y_train,
        cv=cv_for_folds,
        scoring="roc_auc",
        n_jobs=1
    )

    duration = time.time() - start_time

    print(f"\nBest params [{label}]:", best_params)
    print(f"Best validation metrics [{label}]:", best_mets)
    print(f"⏱️ Optimization for [{label}] completed in {duration:.2f} seconds.\n")

    return best_params, best_mets, best_folds, pd.DataFrame(history)


# -----------------------------------------------------------------------------------------
# 6) Run TRUE AOA for the three tree-based models
# -----------------------------------------------------------------------------------------
print("\n>>> Running TRUE AOA Optimization...")

best_params_aoa_brf, best_mets_aoa_brf, folds_aoa_brf, hist_aoa_brf = aoa_optimize_generic(
    build_estimator_fn=build_aoa_brf,
    selected_feature_names=aoa_top_features,
    use_smoteenn=False,
    max_iter=15,
    pop_size=24,
    seed=SEED,
    label="AOA BRF",
    alpha=2.0
)

best_params_aoa_rf_sm, best_mets_aoa_rf_sm, folds_aoa_rf_sm, hist_aoa_rf_sm = aoa_optimize_generic(
    build_estimator_fn=build_aoa_rf,
    selected_feature_names=aoa_top_features,
    use_smoteenn=True,
    max_iter=15,
    pop_size=24,
    seed=SEED,
    label="AOA RF+SMOTEENN",
    alpha=2.0
)

best_params_aoa_brf_sm, best_mets_aoa_brf_sm, folds_aoa_brf_sm, hist_aoa_brf_sm = aoa_optimize_generic(
    build_estimator_fn=build_aoa_brf,
    selected_feature_names=aoa_top_features,
    use_smoteenn=True,
    max_iter=15,
    pop_size=24,
    seed=SEED,
    label="AOA BRF+SMOTEENN",
    alpha=2.0
)


# -----------------------------------------------------------------------------------------
# 7) Final AOA pipelines
# -----------------------------------------------------------------------------------------
pipe_aoa_brf = make_aoa_fs_calibrated_pipeline(
    base_estimator=build_aoa_brf(best_params_aoa_brf),
    selected_feature_names=aoa_top_features,
    sampler=None
)

pipe_aoa_rf_sm = make_aoa_fs_calibrated_pipeline(
    base_estimator=build_aoa_rf(best_params_aoa_rf_sm),
    selected_feature_names=aoa_top_features,
    sampler=SMOTEENN(random_state=SEED)
)

pipe_aoa_brf_sm = make_aoa_fs_calibrated_pipeline(
    base_estimator=build_aoa_brf(best_params_aoa_brf_sm),
    selected_feature_names=aoa_top_features,
    sampler=SMOTEENN(random_state=SEED)
)


aoa_pipelines = {
    "AOA BRF": pipe_aoa_brf,
    "AOA RF+SMOTEENN": pipe_aoa_rf_sm,
    "AOA BRF+SMOTEENN": pipe_aoa_brf_sm
}

aoa_fold_scores = {
    "AOA BRF": folds_aoa_brf,
    "AOA RF+SMOTEENN": folds_aoa_rf_sm,
    "AOA BRF+SMOTEENN": folds_aoa_brf_sm
}


# -----------------------------------------------------------------------------------------
# 8) Baseline-style metric helpers
# -----------------------------------------------------------------------------------------
def compute_binary_metrics_aoa(y_true, y_prob, threshold=0.5):
    y_pred = (y_prob >= threshold).astype(int)

    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "roc_auc": roc_auc_score(y_true, y_prob),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0)
    }


def round_metrics_aoa(mets, ndigits=4):
    return {
        k: round(float(v), ndigits)
        for k, v in mets.items()
    }


def find_best_threshold_aoa(y_true, proba, metric="f1"):
    thresholds = np.linspace(0.01, 0.99, 199)

    best_t = 0.5
    best_val = -np.inf

    for t in thresholds:
        preds = (proba >= t).astype(int)

        if metric == "f1":
            val = f1_score(y_true, preds, zero_division=0)
        elif metric == "recall":
            val = recall_score(y_true, preds, zero_division=0)
        elif metric == "precision":
            val = precision_score(y_true, preds, zero_division=0)
        else:
            val = f1_score(y_true, preds, zero_division=0)

        if val > best_val:
            best_val = val
            best_t = t

    return best_t, best_val


def compute_bootstrap_metrics_with_ci_aoa(
    y_true,
    y_prob,
    threshold,
    n_bootstraps=1000,
    seed=42
):
    rng = np.random.RandomState(seed)

    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)

    boot_aucs = []
    boot_accs = []
    boot_recalls = []
    boot_precisions = []
    boot_f1s = []

    for _ in range(n_bootstraps):
        idx = rng.randint(0, len(y_true), len(y_true))

        if len(np.unique(y_true[idx])) < 2:
            continue

        yt = y_true[idx]
        yp = y_prob[idx]
        pred = (yp >= threshold).astype(int)

        boot_aucs.append(roc_auc_score(yt, yp))
        boot_accs.append(accuracy_score(yt, pred))
        boot_recalls.append(recall_score(yt, pred, zero_division=0))
        boot_precisions.append(precision_score(yt, pred, zero_division=0))
        boot_f1s.append(f1_score(yt, pred, zero_division=0))

    return {
        "auc_ci": np.percentile(boot_aucs, [2.5, 97.5]),
        "acc_ci": np.percentile(boot_accs, [2.5, 97.5]),
        "recall_ci": np.percentile(boot_recalls, [2.5, 97.5]),
        "precision_ci": np.percentile(boot_precisions, [2.5, 97.5]),
        "f1_ci": np.percentile(boot_f1s, [2.5, 97.5])
    }


def format_ci_aoa(value, ci):
    return f"{value:.3f} [{ci[0]:.3f}-{ci[1]:.3f}]"


# -----------------------------------------------------------------------------------------
# 9) Full baseline-style evaluation for AOA models
# -----------------------------------------------------------------------------------------
cv_aoa_eval = StratifiedKFold(
    n_splits=10,
    shuffle=True,
    random_state=SEED
)

aoa_results = []
aoa_model_folds = {}
aoa_fitted_pipelines = {}

print("\n>>> Evaluating TRUE AOA models with baseline-style outputs...")

for name, pipe in aoa_pipelines.items():
    print("\n" + "=" * 95)
    print(f"📈 Evaluating {name}")
    print("=" * 95)

    cooling_pause(
        stage=f"before OOF evaluation for {name}",
        force_short_pause=True
    )

    # Train OOF probability
    proba_train = cross_val_predict(
        pipe,
        X_train,
        y_train,
        cv=cv_aoa_eval,
        method="predict_proba",
        n_jobs=1
    )[:, 1]

    train_05 = compute_binary_metrics_aoa(
        y_train,
        proba_train,
        threshold=0.5
    )

    t_opt, _ = find_best_threshold_aoa(
        y_train,
        proba_train,
        metric="f1"
    )

    train_opt = compute_binary_metrics_aoa(
        y_train,
        proba_train,
        threshold=t_opt
    )
    train_opt["threshold"] = t_opt

    cooling_pause(
        stage=f"before final fit for {name}",
        force_short_pause=True
    )

    # Final fit on full train
    pipe.fit(X_train, y_train)
    aoa_fitted_pipelines[name] = pipe

    cooling_pause(
        stage=f"after final fit for {name}",
        force_short_pause=True
    )

    # Test probability
    proba_test = pipe.predict_proba(X_test)[:, 1]

    test_05 = compute_binary_metrics_aoa(
        y_test,
        proba_test,
        threshold=0.5
    )

    test_opt = compute_binary_metrics_aoa(
        y_test,
        proba_test,
        threshold=t_opt
    )
    test_opt["threshold"] = t_opt

    ci = compute_bootstrap_metrics_with_ci_aoa(
        y_test,
        proba_test,
        threshold=0.5,
        n_bootstraps=1000,
        seed=SEED
    )

    fold_scores = aoa_fold_scores[name]
    aoa_model_folds[name] = fold_scores

    print(f"\n=== {name} Calibrated with PI-TopK ===")
    print("Train (OOF) @0.50:", round_metrics_aoa(train_05))
    print("Train (OOF) @opt :", round_metrics_aoa(train_opt))
    print("Test        @0.50:", round_metrics_aoa(test_05))
    print("Test        @opt :", round_metrics_aoa(test_opt))

    aoa_results.append({
        "Model": name,

        "Train_AUC": train_05["roc_auc"],
        "Train_Accuracy_05": train_05["accuracy"],
        "Train_Recall_05": train_05["recall"],
        "Train_Precision_05": train_05["precision"],
        "Train_F1_05": train_05["f1"],

        "Train_Accuracy_opt": train_opt["accuracy"],
        "Train_Recall_opt": train_opt["recall"],
        "Train_Precision_opt": train_opt["precision"],
        "Train_F1_opt": train_opt["f1"],

        "Test_AUC": test_05["roc_auc"],
        "Test_AUC_95CI": format_ci_aoa(test_05["roc_auc"], ci["auc_ci"]),
        "Test_Accuracy_05": test_05["accuracy"],
        "Test_Accuracy_95CI": format_ci_aoa(test_05["accuracy"], ci["acc_ci"]),
        "Test_Recall_05": test_05["recall"],
        "Test_Recall_95CI": format_ci_aoa(test_05["recall"], ci["recall_ci"]),
        "Test_Precision_05": test_05["precision"],
        "Test_Precision_95CI": format_ci_aoa(test_05["precision"], ci["precision_ci"]),
        "Test_F1_05": test_05["f1"],
        "Test_F1_95CI": format_ci_aoa(test_05["f1"], ci["f1_ci"]),

        "Test_Accuracy_opt": test_opt["accuracy"],
        "Test_Recall_opt": test_opt["recall"],
        "Test_Precision_opt": test_opt["precision"],
        "Test_F1_opt": test_opt["f1"],

        "Optimal_Threshold": t_opt,
        "CV_ROC_AUC_Mean": np.mean(fold_scores),
        "CV_ROC_AUC_Std": np.std(fold_scores)
    })

    # ROC & PR plots — Train OOF
    plt.figure(figsize=(14, 6))

    plt.subplot(1, 2, 1)
    RocCurveDisplay.from_predictions(
        y_train,
        proba_train,
        ax=plt.gca()
    )
    plt.title(f"ROC - Train OOF [{name}]")

    plt.subplot(1, 2, 2)
    PrecisionRecallDisplay.from_predictions(
        y_train,
        proba_train,
        ax=plt.gca()
    )
    plt.title(f"PR - Train OOF [{name}]")

    plt.tight_layout()
    plt.show()

    # ROC & PR plots — Test
    plt.figure(figsize=(14, 6))

    plt.subplot(1, 2, 1)
    RocCurveDisplay.from_predictions(
        y_test,
        proba_test,
        ax=plt.gca()
    )
    plt.title(f"ROC - Test [{name}]")

    plt.subplot(1, 2, 2)
    PrecisionRecallDisplay.from_predictions(
        y_test,
        proba_test,
        ax=plt.gca()
    )
    plt.title(f"PR - Test [{name}]")

    plt.tight_layout()
    plt.show()

    # Confusion matrices @0.50
    pred_train_05 = (proba_train >= 0.5).astype(int)
    pred_test_05 = (proba_test >= 0.5).astype(int)

    plt.figure(figsize=(12, 5))

    plt.subplot(1, 2, 1)
    sns.heatmap(
        confusion_matrix(y_train, pred_train_05),
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=["Pred 0", "Pred 1"],
        yticklabels=["Actual 0", "Actual 1"]
    )
    plt.title(f"Confusion Matrix - Train OOF @0.50 [{name}]")

    plt.subplot(1, 2, 2)
    sns.heatmap(
        confusion_matrix(y_test, pred_test_05),
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=["Pred 0", "Pred 1"],
        yticklabels=["Actual 0", "Actual 1"]
    )
    plt.title(f"Confusion Matrix - Test @0.50 [{name}]")

    plt.tight_layout()
    plt.show()

    # Confusion matrices @opt
    pred_train_opt = (proba_train >= t_opt).astype(int)
    pred_test_opt = (proba_test >= t_opt).astype(int)

    plt.figure(figsize=(12, 5))

    plt.subplot(1, 2, 1)
    sns.heatmap(
        confusion_matrix(y_train, pred_train_opt),
        annot=True,
        fmt="d",
        cmap="Greens",
        xticklabels=["Pred 0", "Pred 1"],
        yticklabels=["Actual 0", "Actual 1"]
    )
    plt.title(f"Confusion Matrix - Train OOF @opt={t_opt:.2f} [{name}]")

    plt.subplot(1, 2, 2)
    sns.heatmap(
        confusion_matrix(y_test, pred_test_opt),
        annot=True,
        fmt="d",
        cmap="Greens",
        xticklabels=["Pred 0", "Pred 1"],
        yticklabels=["Actual 0", "Actual 1"]
    )
    plt.title(f"Confusion Matrix - Test @opt={t_opt:.2f} [{name}]")

    plt.tight_layout()
    plt.show()

    cooling_pause(
        stage=f"between AOA models after {name}",
        force_short_pause=True
    )


# -----------------------------------------------------------------------------------------
# 10) Final AOA summary table
# -----------------------------------------------------------------------------------------
aoa_results_df = pd.DataFrame(aoa_results)

aoa_results_df = aoa_results_df.sort_values(
    by="Test_AUC",
    ascending=False
).reset_index(drop=True)

print("\n" + "=" * 95)
print("📈 TRUE AOA RESULTS — SAME OUTPUT FORMAT AS BASELINE/AOA++")
print("=" * 95)

display(
    aoa_results_df[
        [
            "Model",
            "Test_AUC_95CI",
            "Test_Recall_95CI",
            "Test_F1_95CI",
            "Test_Precision_95CI",
            "Test_Accuracy_95CI",
            "Optimal_Threshold",
            "CV_ROC_AUC_Mean",
            "CV_ROC_AUC_Std"
        ]
    ]
)

aoa_results_df.to_csv(
    "aoa_true_results_same_as_baseline.csv",
    index=False
)

pd.DataFrame(aoa_model_folds).to_csv(
    "aoa_true_10fold_auc_scores.csv",
    index=False
)

hist_aoa_brf.to_csv("aoa_history_brf.csv", index=False)
hist_aoa_rf_sm.to_csv("aoa_history_rf_smoteenn.csv", index=False)
hist_aoa_brf_sm.to_csv("aoa_history_brf_smoteenn.csv", index=False)

print("✅ TRUE AOA analysis completed.")
print("• Saved: aoa_true_results_same_as_baseline.csv")
print("• Saved: aoa_true_10fold_auc_scores.csv")
print("• Saved: aoa_history_brf.csv")
print("• Saved: aoa_history_rf_smoteenn.csv")
print("• Saved: aoa_history_brf_smoteenn.csv")

In [ ]:
# ===== Block 8: Statistical Analysis for AOA-Style Optimized Models =====
# For Cardio Project — AOA BRF, AOA RF+SMOTEENN, AOA BRF+SMOTEENN

import itertools
import scipy.stats as stats
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

print("=" * 95)
print("📊 STEP 4: PAIRWISE STATISTICAL SIGNIFICANCE ANALYSIS (TRUE AOA MODELS)")
print("=" * 95)

# -----------------------------------------------------------------------------------------
# 0) Load saved AOA 10-fold ROC-AUC scores
# Prefer aoa_model_folds from evaluation block.
# Fallback to aoa_fold_scores from optimization block.
# -----------------------------------------------------------------------------------------
if "aoa_model_folds" in globals() and isinstance(aoa_model_folds, dict) and len(aoa_model_folds) > 0:
    aoa_folds = {
        name: np.asarray(scores, dtype=float)
        for name, scores in aoa_model_folds.items()
    }

elif "aoa_fold_scores" in globals() and isinstance(aoa_fold_scores, dict) and len(aoa_fold_scores) > 0:
    aoa_folds = {
        name: np.asarray(scores, dtype=float)
        for name, scores in aoa_fold_scores.items()
    }

else:
    raise NameError(
        "❌ AOA fold scores not found. "
        "Run the TRUE AOA block first. Expected aoa_model_folds or aoa_fold_scores."
    )


# -----------------------------------------------------------------------------------------
# 1) Safety checks
# -----------------------------------------------------------------------------------------
model_names = list(aoa_folds.keys())

print("AOA models included:")
for name in model_names:
    print(f"• {name}: {len(aoa_folds[name])} folds")

fold_lengths = {
    name: len(scores)
    for name, scores in aoa_folds.items()
}

if len(set(fold_lengths.values())) != 1:
    raise ValueError(
        f"❌ Fold count mismatch across AOA models: {fold_lengths}"
    )


# -----------------------------------------------------------------------------------------
# 2) Summary table: mean/std/min/max ROC-AUC
# -----------------------------------------------------------------------------------------
aoa_stat_summary = pd.DataFrame([
    {
        "Model": name,
        "Mean_ROC_AUC": np.mean(scores),
        "Std_ROC_AUC": np.std(scores),
        "Min_ROC_AUC": np.min(scores),
        "Max_ROC_AUC": np.max(scores),
        "Fold_Scores": np.round(scores, 5)
    }
    for name, scores in aoa_folds.items()
]).sort_values(
    by="Mean_ROC_AUC",
    ascending=False
).reset_index(drop=True)

print("\n--- TRUE AOA 10-Fold ROC-AUC Summary ---")
display(aoa_stat_summary)


# -----------------------------------------------------------------------------------------
# 3) Best AOA model based on mean ROC-AUC
# -----------------------------------------------------------------------------------------
best_aoa_model = aoa_stat_summary.iloc[0]["Model"]

print("\n" + "-" * 95)
print(f"Best TRUE AOA model based on mean ROC-AUC: {best_aoa_model}")
print("-" * 95)


# -----------------------------------------------------------------------------------------
# 4) Pairwise paired t-test between all AOA models
# -----------------------------------------------------------------------------------------
pairwise_results = []

print("\n--- Pairwise Paired T-Test Results ---")
print(f"{'Model Pair':<65} | {'T-Stat':<10} | {'P-Value':<10} | {'Significant'}")
print("-" * 105)

for name1, name2 in itertools.combinations(model_names, 2):
    scores1 = aoa_folds[name1]
    scores2 = aoa_folds[name2]

    t_stat, p_val = stats.ttest_rel(scores1, scores2)
    is_sig = "✅ YES" if p_val < 0.05 else "⚠️ NO"

    print(
        f"{name1 + ' vs ' + name2:<65} | "
        f"{t_stat:>10.4f} | "
        f"{p_val:>10.4f} | "
        f"{is_sig}"
    )

    pairwise_results.append({
        "Model_1": name1,
        "Model_2": name2,
        "Mean_1": np.mean(scores1),
        "Mean_2": np.mean(scores2),
        "Mean_Diff": np.mean(scores1) - np.mean(scores2),
        "T_Stat": t_stat,
        "P_Value": p_val,
        "Significant_0.05": p_val < 0.05
    })

aoa_pairwise_ttest_df = pd.DataFrame(pairwise_results)


# -----------------------------------------------------------------------------------------
# 5) Wilcoxon signed-rank test
# Non-parametric paired test
# -----------------------------------------------------------------------------------------
wilcoxon_results = []

print("\n--- Pairwise Wilcoxon Signed-Rank Test Results ---")
print(f"{'Model Pair':<65} | {'W-Stat':<10} | {'P-Value':<10} | {'Significant'}")
print("-" * 105)

for name1, name2 in itertools.combinations(model_names, 2):
    scores1 = aoa_folds[name1]
    scores2 = aoa_folds[name2]

    try:
        w_stat, w_p = stats.wilcoxon(
            scores1,
            scores2,
            zero_method="wilcox",
            alternative="two-sided"
        )
    except ValueError:
        w_stat, w_p = np.nan, np.nan

    is_sig = "✅ YES" if not np.isnan(w_p) and w_p < 0.05 else "⚠️ NO"

    print(
        f"{name1 + ' vs ' + name2:<65} | "
        f"{w_stat:>10.4f} | "
        f"{w_p:>10.4f} | "
        f"{is_sig}"
    )

    wilcoxon_results.append({
        "Model_1": name1,
        "Model_2": name2,
        "Mean_1": np.mean(scores1),
        "Mean_2": np.mean(scores2),
        "Mean_Diff": np.mean(scores1) - np.mean(scores2),
        "W_Stat": w_stat,
        "P_Value": w_p,
        "Significant_0.05": False if np.isnan(w_p) else w_p < 0.05
    })

aoa_wilcoxon_df = pd.DataFrame(wilcoxon_results)


# -----------------------------------------------------------------------------------------
# 6) Stability ranking
# Lower std = more stable
# -----------------------------------------------------------------------------------------
aoa_stability_df = aoa_stat_summary.copy()
aoa_stability_df = aoa_stability_df.sort_values(
    by="Std_ROC_AUC",
    ascending=True
).reset_index(drop=True)

aoa_stability_df["Stability_Rank"] = np.arange(1, len(aoa_stability_df) + 1)

print("\n" + "-" * 95)
print(f"{'AOA Model':<35} | {'Mean ROC-AUC':<12} | {'Std Dev':<10} | {'Stability Rank'}")
print("-" * 95)

for _, row in aoa_stability_df.iterrows():
    print(
        f"{row['Model']:<35} | "
        f"{row['Mean_ROC_AUC']:.5f}      | "
        f"{row['Std_ROC_AUC']:.5f}   | "
        f"{int(row['Stability_Rank'])}"
    )

print("=" * 95 + "\n")


# -----------------------------------------------------------------------------------------
# 7) Boxplot for AOA model stability
# -----------------------------------------------------------------------------------------
plt.figure(figsize=(10, 6))

plot_data = [
    aoa_folds[name]
    for name in model_names
]

try:
    box = plt.boxplot(
        plot_data,
        patch_artist=True,
        tick_labels=model_names,
        widths=0.5
    )
except TypeError:
    box = plt.boxplot(
        plot_data,
        patch_artist=True,
        labels=model_names,
        widths=0.5
    )

colors = ["#2A9D8F", "#E76F51", "#264653"]

for patch, color in zip(box["boxes"], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.8)
    patch.set_edgecolor("black")

for median in box["medians"]:
    median.set_color("#FFB703")
    median.set_linewidth(2)

for whisk in box["whiskers"]:
    whisk.set_color("black")
    whisk.set_linewidth(1.2)

for cap in box["caps"]:
    cap.set_color("black")
    cap.set_linewidth(1.2)

plt.title(
    "Performance Stability & Statistical Distribution of TRUE AOA Models",
    fontsize=13,
    fontweight="bold",
    pad=15
)

plt.ylabel(
    "Validation ROC-AUC Score (10-Fold CV)",
    fontsize=11,
    fontweight="bold"
)

plt.grid(
    True,
    linestyle="--",
    alpha=0.3,
    axis="y"
)

plt.xticks(
    rotation=30,
    ha="right"
)

plt.tight_layout()

plt.savefig(
    "aoa_statistical_comparison.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# -----------------------------------------------------------------------------------------
# 8) Save outputs for paper/report
# -----------------------------------------------------------------------------------------
aoa_stat_summary.to_csv(
    "aoa_statistical_summary.csv",
    index=False
)

aoa_pairwise_ttest_df.to_csv(
    "aoa_pairwise_ttest_results.csv",
    index=False
)

aoa_wilcoxon_df.to_csv(
    "aoa_pairwise_wilcoxon_results.csv",
    index=False
)

aoa_stability_df.to_csv(
    "aoa_stability_ranking.csv",
    index=False
)

pd.DataFrame(aoa_folds).to_csv(
    "aoa_10fold_roc_auc_scores.csv",
    index=False
)

print("✅ Statistical analysis for TRUE AOA models completed successfully.")
print("• Saved: aoa_statistical_comparison.png")
print("• Saved: aoa_statistical_summary.csv")
print("• Saved: aoa_pairwise_ttest_results.csv")
print("• Saved: aoa_pairwise_wilcoxon_results.csv")
print("• Saved: aoa_stability_ranking.csv")
print("• Saved: aoa_10fold_roc_auc_scores.csv")

In [ ]:
# ===== Block 9: Full Pipelines — PI-based FS + TRUE AOA++ Optimization + Calibration + Evaluation =====
# ===== Train/Test Split, Approximate CPU Control, Cooling Pauses, and Evaluation Output =====

import os
import time
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_predict, cross_val_score
from sklearn.metrics import (
    roc_auc_score,
    accuracy_score,
    recall_score,
    precision_score,
    f1_score,
    confusion_matrix,
    RocCurveDisplay,
    PrecisionRecallDisplay
)
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline as SkPipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler, OneHotEncoder

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.combine import SMOTEENN
from imblearn.ensemble import BalancedRandomForestClassifier


print("=" * 95)
print("BLOCK 9: TRUE AOA++ OPTIMIZED MODELS — LARGER CARDIOVASCULAR DATASET")
print("=" * 95)


# -----------------------------------------------------------------------------------------
# 0) Safety checks
# -----------------------------------------------------------------------------------------
required_vars = [
    "X_train", "X_test", "y_train", "y_test",
    "num_cols", "cat_cols",
    "allowed_cores"
]

missing_vars = [v for v in required_vars if v not in globals()]

if missing_vars:
    raise NameError(
        f"❌ Missing variables from previous blocks: {missing_vars}. "
        "Run Blocks 1–5 first."
    )

try:
    SEED
except NameError:
    SEED = 42

random.seed(SEED)
np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

try:
    cooling_pause
except NameError:
    def cooling_pause(stage="", force_short_pause=False):
        if force_short_pause:
            time.sleep(5)

print(f"Allowed CPU cores inside estimators: {allowed_cores}")
print("CV-level n_jobs is set to 1 to avoid nested parallel overload.")


# -----------------------------------------------------------------------------------------
# 1) Use same selected features as baseline
# Baseline block created: global_top_features
# If not available, compute top features only on X_train.
# -----------------------------------------------------------------------------------------
if "global_top_features" in globals():
    top_features = global_top_features
    print("✅ Using global_top_features from exact baseline block.")
else:
    print("⚠️ global_top_features not found. Computing Top-15 features on X_train only...")

    top_features, aoapp_imp_df = compute_perm_importance_topk(
        X_train,
        y_train,
        num_cols,
        cat_cols,
        K=15,
        seed=SEED
    )

print("AOA++ selected features:", top_features)


# -----------------------------------------------------------------------------------------
# 2) Preprocessing and feature selector
# -----------------------------------------------------------------------------------------
def make_aoapp_preprocessor(num_cols, cat_cols):
    num_pipe = SkPipeline([
        ("imp", SimpleImputer(strategy="median")),
        ("scaler", RobustScaler())
    ])

    cat_pipe = SkPipeline([
        ("imp", SimpleImputer(strategy="most_frequent")),
        ("ohe", OneHotEncoder(handle_unknown="ignore"))
    ])

    preprocessor = ColumnTransformer([
        ("num", num_pipe, num_cols),
        ("cat", cat_pipe, cat_cols)
    ])

    return preprocessor


class AOAFeatureNameSelector(BaseEstimator, TransformerMixin):
    def __init__(self, preprocessor=None, selected_feature_names=None):
        self.preprocessor = preprocessor
        self.selected_feature_names = selected_feature_names if selected_feature_names is not None else []

    def fit(self, X, y=None):
        self.preprocessor_ = clone(self.preprocessor)
        self.preprocessor_.fit(X, y)

        all_feats = np.asarray(self.preprocessor_.get_feature_names_out())
        selected = set(map(str, self.selected_feature_names))

        selected_indices = []

        for i, fname in enumerate(all_feats):
            fname = str(fname)
            clean = fname.split("__", 1)[-1]

            keep = False

            if fname in selected or clean in selected:
                keep = True

            for s in selected:
                if clean == s or clean.startswith(s + "_"):
                    keep = True
                    break

            if keep:
                selected_indices.append(i)

        self.feature_names_out_ = all_feats
        self.selected_indices_ = np.array(selected_indices, dtype=int)

        if len(self.selected_indices_) == 0:
            raise ValueError(
                "❌ No selected features matched transformed feature names. "
                "Check top_features, num_cols, cat_cols."
            )

        return self

    def transform(self, X):
        Xt = self.preprocessor_.transform(X)
        return Xt[:, self.selected_indices_]


def make_aoapp_calibrated_classifier(base_estimator):
    try:
        return CalibratedClassifierCV(
            estimator=base_estimator,
            method="isotonic",
            cv=5
        )
    except TypeError:
        return CalibratedClassifierCV(
            base_estimator=base_estimator,
            method="isotonic",
            cv=5
        )


def make_aoapp_fs_calibrated_pipeline(base_estimator, selected_feature_names, sampler=None):
    preprocessor = make_aoapp_preprocessor(
        num_cols=num_cols,
        cat_cols=cat_cols
    )

    feat_sel = AOAFeatureNameSelector(
        preprocessor=preprocessor,
        selected_feature_names=selected_feature_names
    )

    calibrated_model = make_aoapp_calibrated_classifier(base_estimator)

    steps = [("feat_pre", feat_sel)]

    if sampler is not None:
        steps.append(("sampler", sampler))

    steps.append(("model", calibrated_model))

    return ImbPipeline(steps)


# -----------------------------------------------------------------------------------------
# 3) AOA++ search space
# Xi is optimized in normalized continuous space [0, 1]^5.
# Then Xi is decoded to actual RF/BRF hyperparameters.
# -----------------------------------------------------------------------------------------
MAX_DEPTH_CHOICES = [None, 3, 5, 7, 10]
MAX_FEATURES_CHOICES = ["sqrt", "log2", None]


def _decode_int(z, low, high):
    z = float(np.clip(z, 0, 1))
    return int(round(low + z * (high - low)))


def _decode_choice(z, choices):
    z = float(np.clip(z, 0, 1))
    idx = int(round(z * (len(choices) - 1)))
    idx = int(np.clip(idx, 0, len(choices) - 1))
    return choices[idx]


def decode_candidate_vector(x_vec):
    x_vec = np.clip(np.asarray(x_vec, dtype=float), 0, 1)

    return {
        "n_estimators": _decode_int(x_vec[0], 150, 600),
        "max_depth": _decode_choice(x_vec[1], MAX_DEPTH_CHOICES),
        "min_samples_split": _decode_int(x_vec[2], 2, 12),
        "min_samples_leaf": _decode_int(x_vec[3], 1, 6),
        "max_features": _decode_choice(x_vec[4], MAX_FEATURES_CHOICES),
    }


# -----------------------------------------------------------------------------------------
# 4) Estimator builders
# -----------------------------------------------------------------------------------------
def build_aoapp_brf(params):
    return BalancedRandomForestClassifier(
        n_estimators=params["n_estimators"],
        max_depth=params["max_depth"],
        min_samples_split=params["min_samples_split"],
        min_samples_leaf=params["min_samples_leaf"],
        max_features=params["max_features"],
        random_state=SEED,
        n_jobs=allowed_cores
    )


def build_aoapp_rf(params):
    return RandomForestClassifier(
        n_estimators=params["n_estimators"],
        max_depth=params["max_depth"],
        min_samples_split=params["min_samples_split"],
        min_samples_leaf=params["min_samples_leaf"],
        max_features=params["max_features"],
        class_weight="balanced",
        random_state=SEED,
        n_jobs=allowed_cores
    )


# -----------------------------------------------------------------------------------------
# 5) TRUE AOA++ optimizer according to Algorithm 2
# Algorithm minimizes fitness.
# We define fitness = -weighted_score, because we want to maximize score.
# -----------------------------------------------------------------------------------------
def aoapp_optimize_generic(
    build_estimator_fn,
    selected_feature_names,
    use_smoteenn=False,
    max_iter=15,
    pop_size=24,
    seed=42,
    label="GEN",
    alpha_max=2.0,
    gamma=0.1
):
    start_time = time.time()

    rng = np.random.RandomState(seed)
    sampler = SMOTEENN(random_state=seed) if use_smoteenn else None

    # Validation set for AOA++ fitness evaluation
    X_aoa_train, X_aoa_val, y_aoa_train, y_aoa_val = train_test_split(
        X_train,
        y_train,
        test_size=0.2,
        stratify=y_train,
        random_state=seed
    )

    def fitness(x_vec):
        params = decode_candidate_vector(x_vec)

        estimator = build_estimator_fn(params)

        pipe = make_aoapp_fs_calibrated_pipeline(
            base_estimator=estimator,
            selected_feature_names=selected_feature_names,
            sampler=sampler
        )

        pipe.fit(X_aoa_train, y_aoa_train)

        val_proba = pipe.predict_proba(X_aoa_val)[:, 1]
        val_pred = (val_proba >= 0.5).astype(int)

        auc = roc_auc_score(y_aoa_val, val_proba)
        rec = recall_score(y_aoa_val, val_pred, zero_division=0)
        f1 = f1_score(y_aoa_val, val_pred, zero_division=0)
        acc = accuracy_score(y_aoa_val, val_pred)

        score = 0.4 * auc + 0.4 * rec + 0.1 * f1 + 0.1 * acc

        fit_value = -score

        return fit_value, {
            "auc": auc,
            "recall": rec,
            "f1": f1,
            "acc": acc,
            "score": score,
            **params
        }

    # Initialize population Xi for i = 1 to N
    dim = 5
    population = rng.uniform(0, 1, size=(pop_size, dim))

    # Xbest ← initial best solution with lowest fitness
    best_x = None
    best_fit = np.inf
    best_mets = None
    history = []

    for i in range(pop_size):
        fit_i, mets_i = fitness(population[i])

        history.append({
            "iter": 0,
            "individual": i + 1,
            "alpha": np.nan,
            "fitness": fit_i,
            **mets_i
        })

        if fit_i < best_fit:
            best_x = population[i].copy()
            best_fit = fit_i
            best_mets = mets_i.copy()

        if (i + 1) % 6 == 0:
            cooling_pause(
                stage=f"{label} initialization {i+1}/{pop_size}",
                force_short_pause=False
            )

    print(
        f"[{label}] Initial Best | "
        f"Score={best_mets['score']:.4f} | "
        f"AUC={best_mets['auc']:.4f} | "
        f"F1={best_mets['f1']:.4f} | "
        f"ACC={best_mets['acc']:.4f} | "
        f"Recall={best_mets['recall']:.4f}"
    )

    # AOA++ main loop
    for t in range(1, max_iter + 1):

        # Compute XM ← mean of all Xi in population
        XM = np.mean(population, axis=0)

        # alpha ← alpha_max * exp(-gamma * t / T)
        alpha = alpha_max * np.exp(-gamma * t / max_iter)

        new_population = []

        for i in range(pop_size):
            Xi = population[i].copy()

            # rand ← uniform random number in [0, 1]
            rand = rng.uniform(0, 1)

            # X'i ← Xbest + alpha * (rand - 0.5) * (XM - Xi)
            Xi_prime = best_x + alpha * (rand - 0.5) * (XM - Xi)

            # Keep Xi inside normalized bounds
            Xi_prime = np.clip(Xi_prime, 0, 1)

            # Evaluate fitness f(X'i)
            fit_prime, mets_prime = fitness(Xi_prime)

            history.append({
                "iter": t,
                "individual": i + 1,
                "alpha": alpha,
                "fitness": fit_prime,
                **mets_prime
            })

            # if f(X'i) < f(Xbest): Xbest ← X'i
            if fit_prime < best_fit:
                best_x = Xi_prime.copy()
                best_fit = fit_prime
                best_mets = mets_prime.copy()

            # Update position
            new_population.append(Xi_prime)

            if (i + 1) % 6 == 0:
                cooling_pause(
                    stage=f"{label} iter {t}, candidate {i+1}/{pop_size}",
                    force_short_pause=False
                )

        population = np.asarray(new_population)

        print(
            f"[{label}] Iter {t}/{max_iter} | "
            f"Alpha={alpha:.4f} | "
            f"Score={best_mets['score']:.4f} | "
            f"AUC={best_mets['auc']:.4f} | "
            f"F1={best_mets['f1']:.4f} | "
            f"ACC={best_mets['acc']:.4f} | "
            f"Recall={best_mets['recall']:.4f}"
        )

        cooling_pause(
            stage=f"{label} iteration {t}",
            force_short_pause=True
        )

    best_params = decode_candidate_vector(best_x)

    # Fold scores for statistical analysis
    best_pipe = make_aoapp_fs_calibrated_pipeline(
        base_estimator=build_estimator_fn(best_params),
        selected_feature_names=selected_feature_names,
        sampler=sampler
    )

    cv_for_folds = StratifiedKFold(
        n_splits=10,
        shuffle=True,
        random_state=seed
    )

    best_folds = cross_val_score(
        best_pipe,
        X_train,
        y_train,
        cv=cv_for_folds,
        scoring="roc_auc",
        n_jobs=1
    )

    duration = time.time() - start_time

    print(f"\nBest params [{label}]:", best_params)
    print(f"Best validation metrics [{label}]:", best_mets)
    print(f"⏱️ Optimization for [{label}] completed in {duration:.2f} seconds.\n")

    return best_params, best_mets, best_folds, pd.DataFrame(history)


# -----------------------------------------------------------------------------------------
# 6) Run TRUE AOA++ for the three proposed tree-based models
# -----------------------------------------------------------------------------------------
print("\n>>> Running TRUE AOA++ Optimization...")

best_params_aoapp_brf, best_mets_aoapp_brf, folds_aoapp_brf, hist_aoapp_brf = aoapp_optimize_generic(
    build_estimator_fn=build_aoapp_brf,
    selected_feature_names=top_features,
    use_smoteenn=False,
    max_iter=15,
    pop_size=24,
    seed=SEED,
    label="AOA++ BRF",
    alpha_max=2.0,
    gamma=0.1
)

best_params_aoapp_rf_sm, best_mets_aoapp_rf_sm, folds_aoapp_rf_sm, hist_aoapp_rf_sm = aoapp_optimize_generic(
    build_estimator_fn=build_aoapp_rf,
    selected_feature_names=top_features,
    use_smoteenn=True,
    max_iter=15,
    pop_size=24,
    seed=SEED,
    label="AOA++ RF+SMOTEENN",
    alpha_max=2.0,
    gamma=0.1
)

best_params_aoapp_brf_sm, best_mets_aoapp_brf_sm, folds_aoapp_brf_sm, hist_aoapp_brf_sm = aoapp_optimize_generic(
    build_estimator_fn=build_aoapp_brf,
    selected_feature_names=top_features,
    use_smoteenn=True,
    max_iter=15,
    pop_size=24,
    seed=SEED,
    label="AOA++ BRF+SMOTEENN",
    alpha_max=2.0,
    gamma=0.1
)


# -----------------------------------------------------------------------------------------
# 7) Final AOA++ pipelines
# -----------------------------------------------------------------------------------------
pipe_aoapp_brf = make_aoapp_fs_calibrated_pipeline(
    base_estimator=build_aoapp_brf(best_params_aoapp_brf),
    selected_feature_names=top_features,
    sampler=None
)

pipe_aoapp_rf_sm = make_aoapp_fs_calibrated_pipeline(
    base_estimator=build_aoapp_rf(best_params_aoapp_rf_sm),
    selected_feature_names=top_features,
    sampler=SMOTEENN(random_state=SEED)
)

pipe_aoapp_brf_sm = make_aoapp_fs_calibrated_pipeline(
    base_estimator=build_aoapp_brf(best_params_aoapp_brf_sm),
    selected_feature_names=top_features,
    sampler=SMOTEENN(random_state=SEED)
)


aoapp_pipelines = {
    "AOA++ BRF": pipe_aoapp_brf,
    "AOA++ RF+SMOTEENN": pipe_aoapp_rf_sm,
    "AOA++ BRF+SMOTEENN": pipe_aoapp_brf_sm
}

aoapp_fold_scores = {
    "AOA++ BRF": folds_aoapp_brf,
    "AOA++ RF+SMOTEENN": folds_aoapp_rf_sm,
    "AOA++ BRF+SMOTEENN": folds_aoapp_brf_sm
}


# -----------------------------------------------------------------------------------------
# 8) Baseline-style metric helpers
# -----------------------------------------------------------------------------------------
def compute_binary_metrics_aoapp(y_true, y_prob, threshold=0.5):
    y_pred = (y_prob >= threshold).astype(int)

    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "roc_auc": roc_auc_score(y_true, y_prob),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0)
    }


def round_metrics_aoapp(mets, ndigits=4):
    return {
        k: round(v, ndigits)
        for k, v in mets.items()
    }


def find_best_threshold_aoapp(y_true, proba, metric="f1"):
    thresholds = np.linspace(0.01, 0.99, 199)

    best_t = 0.5
    best_val = -np.inf

    for t in thresholds:
        preds = (proba >= t).astype(int)

        if metric == "f1":
            val = f1_score(y_true, preds, zero_division=0)
        elif metric == "recall":
            val = recall_score(y_true, preds, zero_division=0)
        elif metric == "precision":
            val = precision_score(y_true, preds, zero_division=0)
        else:
            val = f1_score(y_true, preds, zero_division=0)

        if val > best_val:
            best_val = val
            best_t = t

    return best_t, best_val


def compute_bootstrap_metrics_with_ci_aoapp(
    y_true,
    y_prob,
    threshold,
    n_bootstraps=1000,
    seed=42
):
    rng = np.random.RandomState(seed)

    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)

    boot_aucs = []
    boot_accs = []
    boot_recalls = []
    boot_precisions = []
    boot_f1s = []

    for _ in range(n_bootstraps):
        idx = rng.randint(0, len(y_true), len(y_true))

        if len(np.unique(y_true[idx])) < 2:
            continue

        yt = y_true[idx]
        yp = y_prob[idx]
        pred = (yp >= threshold).astype(int)

        boot_aucs.append(roc_auc_score(yt, yp))
        boot_accs.append(accuracy_score(yt, pred))
        boot_recalls.append(recall_score(yt, pred, zero_division=0))
        boot_precisions.append(precision_score(yt, pred, zero_division=0))
        boot_f1s.append(f1_score(yt, pred, zero_division=0))

    return {
        "auc_ci": np.percentile(boot_aucs, [2.5, 97.5]),
        "acc_ci": np.percentile(boot_accs, [2.5, 97.5]),
        "recall_ci": np.percentile(boot_recalls, [2.5, 97.5]),
        "precision_ci": np.percentile(boot_precisions, [2.5, 97.5]),
        "f1_ci": np.percentile(boot_f1s, [2.5, 97.5])
    }


def format_ci_aoapp(value, ci):
    return f"{value:.3f} [{ci[0]:.3f}-{ci[1]:.3f}]"


# -----------------------------------------------------------------------------------------
# 9) Full baseline-style evaluation for AOA++ models
# -----------------------------------------------------------------------------------------
cv_aoapp_eval = StratifiedKFold(
    n_splits=10,
    shuffle=True,
    random_state=SEED
)

aoapp_results = []
aoapp_model_folds = {}
aoapp_fitted_pipelines = {}

print("\n>>> Evaluating TRUE AOA++ models with baseline-style outputs...")

for name, pipe in aoapp_pipelines.items():
    print("\n" + "=" * 95)
    print(f"📈 Evaluating {name}")
    print("=" * 95)

    cooling_pause(
        stage=f"before OOF evaluation for {name}",
        force_short_pause=True
    )

    # Train OOF probability
    proba_train = cross_val_predict(
        pipe,
        X_train,
        y_train,
        cv=cv_aoapp_eval,
        method="predict_proba",
        n_jobs=1
    )[:, 1]

    train_05 = compute_binary_metrics_aoapp(
        y_train,
        proba_train,
        threshold=0.5
    )

    t_opt, _ = find_best_threshold_aoapp(
        y_train,
        proba_train,
        metric="f1"
    )

    train_opt = compute_binary_metrics_aoapp(
        y_train,
        proba_train,
        threshold=t_opt
    )
    train_opt["threshold"] = t_opt

    cooling_pause(
        stage=f"before final fit for {name}",
        force_short_pause=True
    )

    # Final fit on full train
    pipe.fit(X_train, y_train)
    aoapp_fitted_pipelines[name] = pipe

    cooling_pause(
        stage=f"after final fit for {name}",
        force_short_pause=True
    )

    # Test probability
    proba_test = pipe.predict_proba(X_test)[:, 1]

    test_05 = compute_binary_metrics_aoapp(
        y_test,
        proba_test,
        threshold=0.5
    )

    test_opt = compute_binary_metrics_aoapp(
        y_test,
        proba_test,
        threshold=t_opt
    )
    test_opt["threshold"] = t_opt

    ci = compute_bootstrap_metrics_with_ci_aoapp(
        y_test,
        proba_test,
        threshold=0.5,
        n_bootstraps=1000,
        seed=SEED
    )

    fold_scores = aoapp_fold_scores[name]
    aoapp_model_folds[name] = fold_scores

    print(f"\n=== {name} Calibrated with PI-TopK ===")
    print("Train (OOF) @0.50:", round_metrics_aoapp(train_05))
    print("Train (OOF) @opt :", round_metrics_aoapp(train_opt))
    print("Test        @0.50:", round_metrics_aoapp(test_05))
    print("Test        @opt :", round_metrics_aoapp(test_opt))

    aoapp_results.append({
        "Model": name,

        "Train_AUC": train_05["roc_auc"],
        "Train_Accuracy_05": train_05["accuracy"],
        "Train_Recall_05": train_05["recall"],
        "Train_Precision_05": train_05["precision"],
        "Train_F1_05": train_05["f1"],

        "Train_Accuracy_opt": train_opt["accuracy"],
        "Train_Recall_opt": train_opt["recall"],
        "Train_Precision_opt": train_opt["precision"],
        "Train_F1_opt": train_opt["f1"],

        "Test_AUC": test_05["roc_auc"],
        "Test_AUC_95CI": format_ci_aoapp(test_05["roc_auc"], ci["auc_ci"]),
        "Test_Accuracy_05": test_05["accuracy"],
        "Test_Accuracy_95CI": format_ci_aoapp(test_05["accuracy"], ci["acc_ci"]),
        "Test_Recall_05": test_05["recall"],
        "Test_Recall_95CI": format_ci_aoapp(test_05["recall"], ci["recall_ci"]),
        "Test_Precision_05": test_05["precision"],
        "Test_Precision_95CI": format_ci_aoapp(test_05["precision"], ci["precision_ci"]),
        "Test_F1_05": test_05["f1"],
        "Test_F1_95CI": format_ci_aoapp(test_05["f1"], ci["f1_ci"]),

        "Test_Accuracy_opt": test_opt["accuracy"],
        "Test_Recall_opt": test_opt["recall"],
        "Test_Precision_opt": test_opt["precision"],
        "Test_F1_opt": test_opt["f1"],

        "Optimal_Threshold": t_opt,
        "CV_ROC_AUC_Mean": np.mean(fold_scores),
        "CV_ROC_AUC_Std": np.std(fold_scores)
    })

    # ROC & PR plots — Train OOF
    plt.figure(figsize=(14, 6))

    plt.subplot(1, 2, 1)
    RocCurveDisplay.from_predictions(
        y_train,
        proba_train,
        ax=plt.gca()
    )
    plt.title(f"ROC - Train OOF [{name}]")

    plt.subplot(1, 2, 2)
    PrecisionRecallDisplay.from_predictions(
        y_train,
        proba_train,
        ax=plt.gca()
    )
    plt.title(f"PR - Train OOF [{name}]")

    plt.tight_layout()
    plt.show()

    # ROC & PR plots — Test
    plt.figure(figsize=(14, 6))

    plt.subplot(1, 2, 1)
    RocCurveDisplay.from_predictions(
        y_test,
        proba_test,
        ax=plt.gca()
    )
    plt.title(f"ROC - Test [{name}]")

    plt.subplot(1, 2, 2)
    PrecisionRecallDisplay.from_predictions(
        y_test,
        proba_test,
        ax=plt.gca()
    )
    plt.title(f"PR - Test [{name}]")

    plt.tight_layout()
    plt.show()

    # Confusion matrices @0.50
    pred_train_05 = (proba_train >= 0.5).astype(int)
    pred_test_05 = (proba_test >= 0.5).astype(int)

    plt.figure(figsize=(12, 5))

    plt.subplot(1, 2, 1)
    sns.heatmap(
        confusion_matrix(y_train, pred_train_05),
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=["Pred 0", "Pred 1"],
        yticklabels=["Actual 0", "Actual 1"]
    )
    plt.title(f"Confusion Matrix - Train OOF @0.50 [{name}]")

    plt.subplot(1, 2, 2)
    sns.heatmap(
        confusion_matrix(y_test, pred_test_05),
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=["Pred 0", "Pred 1"],
        yticklabels=["Actual 0", "Actual 1"]
    )
    plt.title(f"Confusion Matrix - Test @0.50 [{name}]")

    plt.tight_layout()
    plt.show()

    # Confusion matrices @opt
    pred_train_opt = (proba_train >= t_opt).astype(int)
    pred_test_opt = (proba_test >= t_opt).astype(int)

    plt.figure(figsize=(12, 5))

    plt.subplot(1, 2, 1)
    sns.heatmap(
        confusion_matrix(y_train, pred_train_opt),
        annot=True,
        fmt="d",
        cmap="Greens",
        xticklabels=["Pred 0", "Pred 1"],
        yticklabels=["Actual 0", "Actual 1"]
    )
    plt.title(f"Confusion Matrix - Train OOF @opt={t_opt:.2f} [{name}]")

    plt.subplot(1, 2, 2)
    sns.heatmap(
        confusion_matrix(y_test, pred_test_opt),
        annot=True,
        fmt="d",
        cmap="Greens",
        xticklabels=["Pred 0", "Pred 1"],
        yticklabels=["Actual 0", "Actual 1"]
    )
    plt.title(f"Confusion Matrix - Test @opt={t_opt:.2f} [{name}]")

    plt.tight_layout()
    plt.show()

    cooling_pause(
        stage=f"between AOA++ models after {name}",
        force_short_pause=True
    )


# -----------------------------------------------------------------------------------------
# 10) Final AOA++ summary table
# -----------------------------------------------------------------------------------------
aoapp_results_df = pd.DataFrame(aoapp_results)

aoapp_results_df = aoapp_results_df.sort_values(
    by="Test_AUC",
    ascending=False
).reset_index(drop=True)

print("\n" + "=" * 95)
print("📈 TRUE AOA++ RESULTS — SAME OUTPUT FORMAT AS BASELINE")
print("=" * 95)

display(
    aoapp_results_df[
        [
            "Model",
            "Test_AUC_95CI",
            "Test_Recall_95CI",
            "Test_F1_95CI",
            "Test_Precision_95CI",
            "Test_Accuracy_95CI",
            "Optimal_Threshold",
            "CV_ROC_AUC_Mean",
            "CV_ROC_AUC_Std"
        ]
    ]
)

aoapp_results_df.to_csv(
    "aoapp_true_results_same_as_baseline.csv",
    index=False
)

pd.DataFrame(aoapp_model_folds).to_csv(
    "aoapp_true_10fold_auc_scores.csv",
    index=False
)

hist_aoapp_brf.to_csv("aoapp_history_brf.csv", index=False)
hist_aoapp_rf_sm.to_csv("aoapp_history_rf_smoteenn.csv", index=False)
hist_aoapp_brf_sm.to_csv("aoapp_history_brf_smoteenn.csv", index=False)

print("✅ TRUE AOA++ analysis completed.")
print("• Saved: aoapp_true_results_same_as_baseline.csv")
print("• Saved: aoapp_true_10fold_auc_scores.csv")
print("• Saved: aoapp_history_brf.csv")
print("• Saved: aoapp_history_rf_smoteenn.csv")
print("• Saved: aoapp_history_brf_smoteenn.csv")

In [ ]:
# ===== Block 10: Pairwise Statistical Comparison of TRUE AOA++ Models =====
# For Cardio Project — AOA++ BRF, AOA++ RF+SMOTEENN, AOA++ BRF+SMOTEENN

import itertools
import scipy.stats as stats
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

print("=" * 95)
print("📊 STEP 4: PAIRWISE STATISTICAL SIGNIFICANCE ANALYSIS (TRUE AOA++ MODELS)")
print("=" * 95)

# -----------------------------------------------------------------------------------------
# 0) Load saved AOA++ 10-fold ROC-AUC scores
# Prefer aoapp_model_folds from evaluation block.
# Fallback to aoapp_fold_scores from optimization block.
# -----------------------------------------------------------------------------------------
if "aoapp_model_folds" in globals() and isinstance(aoapp_model_folds, dict) and len(aoapp_model_folds) > 0:
    aoapp_folds = {
        name: np.asarray(scores, dtype=float)
        for name, scores in aoapp_model_folds.items()
    }

elif "aoapp_fold_scores" in globals() and isinstance(aoapp_fold_scores, dict) and len(aoapp_fold_scores) > 0:
    aoapp_folds = {
        name: np.asarray(scores, dtype=float)
        for name, scores in aoapp_fold_scores.items()
    }

else:
    raise NameError(
        "❌ AOA++ fold scores not found. "
        "Run the TRUE AOA++ block first. Expected aoapp_model_folds or aoapp_fold_scores."
    )


# -----------------------------------------------------------------------------------------
# 1) Safety checks
# -----------------------------------------------------------------------------------------
model_names = list(aoapp_folds.keys())

print("AOA++ models included:")
for name in model_names:
    print(f"• {name}: {len(aoapp_folds[name])} folds")

fold_lengths = {
    name: len(scores)
    for name, scores in aoapp_folds.items()
}

if len(set(fold_lengths.values())) != 1:
    raise ValueError(
        f"❌ Fold count mismatch across AOA++ models: {fold_lengths}"
    )


# -----------------------------------------------------------------------------------------
# 2) Summary table: mean/std/min/max ROC-AUC
# -----------------------------------------------------------------------------------------
aoapp_stat_summary = pd.DataFrame([
    {
        "Model": name,
        "Mean_ROC_AUC": np.mean(scores),
        "Std_ROC_AUC": np.std(scores),
        "Min_ROC_AUC": np.min(scores),
        "Max_ROC_AUC": np.max(scores),
        "Fold_Scores": np.round(scores, 5)
    }
    for name, scores in aoapp_folds.items()
]).sort_values(
    by="Mean_ROC_AUC",
    ascending=False
).reset_index(drop=True)

print("\n--- TRUE AOA++ 10-Fold ROC-AUC Summary ---")
display(aoapp_stat_summary)


# -----------------------------------------------------------------------------------------
# 3) Best AOA++ model based on mean ROC-AUC
# -----------------------------------------------------------------------------------------
best_aoapp_model = aoapp_stat_summary.iloc[0]["Model"]

print("\n" + "-" * 95)
print(f"Best TRUE AOA++ model based on mean ROC-AUC: {best_aoapp_model}")
print("-" * 95)


# -----------------------------------------------------------------------------------------
# 4) Pairwise paired t-test between all AOA++ models
# -----------------------------------------------------------------------------------------
pairwise_results = []

print("\n--- Pairwise Paired T-Test Results ---")
print(f"{'Model Pair':<65} | {'T-Stat':<10} | {'P-Value':<10} | {'Significant'}")
print("-" * 105)

for name1, name2 in itertools.combinations(model_names, 2):
    scores1 = aoapp_folds[name1]
    scores2 = aoapp_folds[name2]

    t_stat, p_val = stats.ttest_rel(scores1, scores2)
    is_sig = "✅ YES" if p_val < 0.05 else "⚠️ NO"

    print(
        f"{name1 + ' vs ' + name2:<65} | "
        f"{t_stat:>10.4f} | "
        f"{p_val:>10.4f} | "
        f"{is_sig}"
    )

    pairwise_results.append({
        "Model_1": name1,
        "Model_2": name2,
        "Mean_1": np.mean(scores1),
        "Mean_2": np.mean(scores2),
        "Mean_Diff": np.mean(scores1) - np.mean(scores2),
        "T_Stat": t_stat,
        "P_Value": p_val,
        "Significant_0.05": p_val < 0.05
    })

aoapp_pairwise_ttest_df = pd.DataFrame(pairwise_results)


# -----------------------------------------------------------------------------------------
# 5) Wilcoxon signed-rank test
# More conservative non-parametric paired test
# -----------------------------------------------------------------------------------------
wilcoxon_results = []

print("\n--- Pairwise Wilcoxon Signed-Rank Test Results ---")
print(f"{'Model Pair':<65} | {'W-Stat':<10} | {'P-Value':<10} | {'Significant'}")
print("-" * 105)

for name1, name2 in itertools.combinations(model_names, 2):
    scores1 = aoapp_folds[name1]
    scores2 = aoapp_folds[name2]

    try:
        w_stat, w_p = stats.wilcoxon(
            scores1,
            scores2,
            zero_method="wilcox",
            alternative="two-sided"
        )
    except ValueError:
        w_stat, w_p = np.nan, np.nan

    is_sig = "✅ YES" if not np.isnan(w_p) and w_p < 0.05 else "⚠️ NO"

    print(
        f"{name1 + ' vs ' + name2:<65} | "
        f"{w_stat:>10.4f} | "
        f"{w_p:>10.4f} | "
        f"{is_sig}"
    )

    wilcoxon_results.append({
        "Model_1": name1,
        "Model_2": name2,
        "Mean_1": np.mean(scores1),
        "Mean_2": np.mean(scores2),
        "Mean_Diff": np.mean(scores1) - np.mean(scores2),
        "W_Stat": w_stat,
        "P_Value": w_p,
        "Significant_0.05": False if np.isnan(w_p) else w_p < 0.05
    })

aoapp_wilcoxon_df = pd.DataFrame(wilcoxon_results)


# -----------------------------------------------------------------------------------------
# 6) Stability ranking
# Lower std = more stable
# -----------------------------------------------------------------------------------------
aoapp_stability_df = aoapp_stat_summary.copy()
aoapp_stability_df = aoapp_stability_df.sort_values(
    by="Std_ROC_AUC",
    ascending=True
).reset_index(drop=True)

aoapp_stability_df["Stability_Rank"] = np.arange(1, len(aoapp_stability_df) + 1)

print("\n" + "-" * 95)
print(f"{'AOA++ Model':<35} | {'Mean ROC-AUC':<12} | {'Std Dev':<10} | {'Stability Rank'}")
print("-" * 95)

for _, row in aoapp_stability_df.iterrows():
    print(
        f"{row['Model']:<35} | "
        f"{row['Mean_ROC_AUC']:.5f}      | "
        f"{row['Std_ROC_AUC']:.5f}   | "
        f"{int(row['Stability_Rank'])}"
    )

print("=" * 95 + "\n")


# -----------------------------------------------------------------------------------------
# 7) Boxplot for AOA++ model stability
# -----------------------------------------------------------------------------------------
plt.figure(figsize=(10, 6))

plot_data = [
    aoapp_folds[name]
    for name in model_names
]

try:
    box = plt.boxplot(
        plot_data,
        patch_artist=True,
        tick_labels=model_names,
        widths=0.5
    )
except TypeError:
    box = plt.boxplot(
        plot_data,
        patch_artist=True,
        labels=model_names,
        widths=0.5
    )

colors = ["#1D3557", "#E63946", "#457B9D"]

for patch, color in zip(box["boxes"], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.8)
    patch.set_edgecolor("black")

for median in box["medians"]:
    median.set_color("#FFB703")
    median.set_linewidth(2)

for whisk in box["whiskers"]:
    whisk.set_color("black")
    whisk.set_linewidth(1.2)

for cap in box["caps"]:
    cap.set_color("black")
    cap.set_linewidth(1.2)

plt.title(
    "Performance Stability & Statistical Distribution of TRUE AOA++ Models",
    fontsize=13,
    fontweight="bold",
    pad=15
)

plt.ylabel(
    "Validation ROC-AUC Score (10-Fold CV)",
    fontsize=11,
    fontweight="bold"
)

plt.grid(
    True,
    linestyle="--",
    alpha=0.3,
    axis="y"
)

plt.xticks(
    rotation=30,
    ha="right"
)

plt.tight_layout()

plt.savefig(
    "aoapp_statistical_comparison.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# -----------------------------------------------------------------------------------------
# 8) Save outputs for paper/report
# -----------------------------------------------------------------------------------------
aoapp_stat_summary.to_csv(
    "aoapp_statistical_summary.csv",
    index=False
)

aoapp_pairwise_ttest_df.to_csv(
    "aoapp_pairwise_ttest_results.csv",
    index=False
)

aoapp_wilcoxon_df.to_csv(
    "aoapp_pairwise_wilcoxon_results.csv",
    index=False
)

aoapp_stability_df.to_csv(
    "aoapp_stability_ranking.csv",
    index=False
)

pd.DataFrame(aoapp_folds).to_csv(
    "aoapp_10fold_roc_auc_scores.csv",
    index=False
)

print("✅ Statistical analysis for TRUE AOA++ models completed successfully.")
print("• Saved: aoapp_statistical_comparison.png")
print("• Saved: aoapp_statistical_summary.csv")
print("• Saved: aoapp_pairwise_ttest_results.csv")
print("• Saved: aoapp_pairwise_wilcoxon_results.csv")
print("• Saved: aoapp_stability_ranking.csv")
print("• Saved: aoapp_10fold_roc_auc_scores.csv")

In [ ]:
# ===== Block 11: SHAP Analysis and Feature Importance Visualization =====
# Reviewer-ready version for the second Cardio dataset:
# 1) Explains the final fitted calibrated AOA++ BRF model
# 2) Computes Mean/Std SHAP across training samples
# 3) Computes SHAP feature-ranking stability across stratified CV folds
# 4) Saves numerical stability tables and publication-ready figures

import os
import time
import psutil
import shap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import sparse

from sklearn.base import clone
from sklearn.model_selection import StratifiedKFold

print("\n>>> Starting Fold-Stability-Validated SHAP Analysis for TRUE AOA++ Models...")


# -----------------------------------------------------------------------------------------
# 0) CPU / cooling compatibility
# -----------------------------------------------------------------------------------------
try:
    allowed_cores
except NameError:
    physical_cores = psutil.cpu_count(logical=False) or psutil.cpu_count(logical=True) or 1
    allowed_cores = max(1, int(physical_cores * 0.8))

os.environ["OMP_NUM_THREADS"] = str(allowed_cores)
os.environ["MKL_NUM_THREADS"] = str(allowed_cores)
os.environ["OPENBLAS_NUM_THREADS"] = str(allowed_cores)
os.environ["NUMEXPR_NUM_THREADS"] = str(allowed_cores)

try:
    COOL_DOWN_SECONDS
except NameError:
    COOL_DOWN_SECONDS = 5

RANDOM_STATE_SHAP = 42
SHAP_STABILITY_N_SPLITS = 5
SHAP_STABILITY_TOP_N = 10

# For the larger Cardio dataset, SHAP can be computationally heavy.
# Keep None to use all validation samples in each fold.
# If runtime becomes too long, set this to an integer such as 2000.
SHAP_MAX_SAMPLES_PER_FOLD = None


# -----------------------------------------------------------------------------------------
# 1) Select final fitted pipeline
# Default: best TRUE AOA++ BRF pipeline for current Cardio dataset
# You can change this to:
# final_pipe = pipe_aoapp_rf_sm
# final_pipe = pipe_aoapp_brf_sm
# -----------------------------------------------------------------------------------------
final_pipe = pipe_aoapp_brf
final_model_name = "TRUE AOA++ BRF"


# -----------------------------------------------------------------------------------------
# 2) Utility functions
# -----------------------------------------------------------------------------------------
def safe_index_rows(X, idx):
    if hasattr(X, "iloc"):
        return X.iloc[idx].copy()
    return np.asarray(X)[idx]


def safe_index_vector(y, idx):
    if hasattr(y, "iloc"):
        return y.iloc[idx].copy()
    return np.asarray(y)[idx]


def to_dense_array(X):
    if sparse.issparse(X):
        return X.toarray()
    return np.asarray(X)


def make_unique_feature_names(names):
    counts = {}
    unique_names = []

    for name in names:
        name = str(name)

        if name not in counts:
            counts[name] = 0
            unique_names.append(name)
        else:
            counts[name] += 1
            unique_names.append(f"{name}_{counts[name]}")

    return unique_names


def get_processed_feature_names(feat_selector, X_processed):
    feature_names = None

    if hasattr(feat_selector, "feature_names_out_") and hasattr(feat_selector, "selected_indices_"):
        all_names = np.asarray(feat_selector.feature_names_out_)
        selected_idx = np.asarray(feat_selector.selected_indices_)
        feature_names = all_names[selected_idx]

    elif hasattr(feat_selector, "get_feature_names_out"):
        try:
            feature_names = feat_selector.get_feature_names_out()
        except Exception:
            feature_names = None

    if feature_names is None or len(feature_names) != X_processed.shape[1]:
        feature_names = [f"feature_{i}" for i in range(X_processed.shape[1])]
    else:
        feature_names = [
            str(f).split("__", 1)[-1]
            for f in feature_names
        ]

    return make_unique_feature_names(feature_names)


def extract_base_estimator(calibrated_clf):
    if hasattr(calibrated_clf, "estimator"):
        return calibrated_clf.estimator

    if hasattr(calibrated_clf, "base_estimator"):
        return calibrated_clf.base_estimator

    if hasattr(calibrated_clf, "classifier"):
        return calibrated_clf.classifier

    raise AttributeError(
        "Could not extract base estimator from calibrated classifier. "
        "Check your sklearn version."
    )


def get_class1_shap_values(raw_shap_values):
    if isinstance(raw_shap_values, list):
        return raw_shap_values[1]

    raw_shap_values = np.asarray(raw_shap_values)

    if raw_shap_values.ndim == 3:
        # Common newer SHAP format: samples × features × classes
        if raw_shap_values.shape[2] == 2:
            return raw_shap_values[:, :, 1]

        # Alternative format: classes × samples × features
        if raw_shap_values.shape[0] == 2:
            return raw_shap_values[1, :, :]

    if raw_shap_values.ndim == 2:
        return raw_shap_values

    raise ValueError(
        f"Unsupported SHAP values shape: {raw_shap_values.shape}"
    )


def compute_shap_for_fitted_pipeline(fitted_pipe, X_input, context_name=""):
    feat_step = fitted_pipe.named_steps["feat_pre"]

    X_processed = feat_step.transform(X_input)
    X_processed = to_dense_array(X_processed)

    feature_names = get_processed_feature_names(feat_step, X_processed)

    X_shap_df = pd.DataFrame(
        X_processed,
        columns=feature_names
    )

    calibrated_model = fitted_pipe.named_steps["model"]
    calibrated_classifiers = calibrated_model.calibrated_classifiers_

    base_estimators = [
        extract_base_estimator(cc)
        for cc in calibrated_classifiers
    ]

    print(
        f"{context_name} | SHAP input shape: {X_shap_df.shape} | "
        f"Base estimators: {len(base_estimators)}"
    )

    all_shap_values = []

    for idx, base_estimator in enumerate(base_estimators, start=1):
        print(
            f"{context_name} | Computing SHAP for calibrated estimator "
            f"{idx}/{len(base_estimators)}..."
        )

        explainer = shap.TreeExplainer(base_estimator)
        raw_shap_values = explainer.shap_values(X_shap_df)

        shap_vals_class1_i = get_class1_shap_values(raw_shap_values)

        if shap_vals_class1_i.shape[1] != X_shap_df.shape[1]:
            raise ValueError(
                f"SHAP feature mismatch in {context_name}: "
                f"SHAP has {shap_vals_class1_i.shape[1]} features, "
                f"but X has {X_shap_df.shape[1]} features."
            )

        all_shap_values.append(shap_vals_class1_i)

        if COOL_DOWN_SECONDS > 0:
            time.sleep(COOL_DOWN_SECONDS)

    shap_vals_class1 = np.mean(
        np.stack(all_shap_values, axis=0),
        axis=0
    )

    return shap_vals_class1, X_shap_df, feature_names


# -----------------------------------------------------------------------------------------
# 3) Make sure final pipeline is fitted
# -----------------------------------------------------------------------------------------
try:
    _ = final_pipe.named_steps["model"].calibrated_classifiers_
except Exception:
    print("Pipeline was not fitted. Fitting final pipeline now...")
    final_pipe.fit(X_train, y_train)


# -----------------------------------------------------------------------------------------
# 4) Final fitted model SHAP analysis on processed training data
# -----------------------------------------------------------------------------------------
shap_vals_class1, X_shap_df, feature_names = compute_shap_for_fitted_pipeline(
    final_pipe,
    X_train,
    context_name="Final fitted pipeline"
)

print(f"Final SHAP input shape: {X_shap_df.shape}")
print(f"Number of final SHAP features: {len(feature_names)}")


# -----------------------------------------------------------------------------------------
# 5) Sample-level SHAP variation for final fitted model
# Mean absolute SHAP = global importance
# Std absolute SHAP = variation across training samples
# -----------------------------------------------------------------------------------------
mean_abs_shap = np.mean(
    np.abs(shap_vals_class1),
    axis=0
)

std_abs_shap = np.std(
    np.abs(shap_vals_class1),
    axis=0
)

stability_df = pd.DataFrame({
    "Feature": feature_names,
    "Mean_SHAP": mean_abs_shap,
    "Std_SHAP_AcrossSamples": std_abs_shap
}).sort_values(
    by="Mean_SHAP",
    ascending=False
).reset_index(drop=True)

stability_df["FinalModel_Rank"] = np.arange(1, len(stability_df) + 1)

print("\n--- Final Model SHAP Importance and Across-Sample Variation ---")
print(stability_df)


# -----------------------------------------------------------------------------------------
# 6) Fold-level SHAP feature-ranking stability
# This directly addresses the reviewer concern about stability/reproducibility
# of feature rankings across multiple folds.
# -----------------------------------------------------------------------------------------
print("\n>>> Starting fold-level SHAP ranking stability analysis...")

y_train_array = np.asarray(y_train)

cv = StratifiedKFold(
    n_splits=SHAP_STABILITY_N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE_SHAP
)

fold_importance_tables = []

for fold_id, (train_idx, valid_idx) in enumerate(cv.split(X_train, y_train_array), start=1):
    print(f"\n--- SHAP stability fold {fold_id}/{SHAP_STABILITY_N_SPLITS} ---")

    X_fold_train = safe_index_rows(X_train, train_idx)
    y_fold_train = safe_index_vector(y_train, train_idx)

    X_fold_valid = safe_index_rows(X_train, valid_idx)

    if SHAP_MAX_SAMPLES_PER_FOLD is not None and len(X_fold_valid) > SHAP_MAX_SAMPLES_PER_FOLD:
        rng = np.random.RandomState(RANDOM_STATE_SHAP + fold_id)
        sampled_positions = rng.choice(
            np.arange(len(X_fold_valid)),
            size=SHAP_MAX_SAMPLES_PER_FOLD,
            replace=False
        )
        X_fold_valid = safe_index_rows(X_fold_valid, sampled_positions)

    fold_pipe = clone(final_pipe)
    fold_pipe.fit(X_fold_train, y_fold_train)

    fold_shap_vals, fold_X_shap_df, fold_feature_names = compute_shap_for_fitted_pipeline(
        fold_pipe,
        X_fold_valid,
        context_name=f"Fold {fold_id}"
    )

    fold_mean_abs_shap = np.mean(
        np.abs(fold_shap_vals),
        axis=0
    )

    fold_df = pd.DataFrame({
        "Fold": fold_id,
        "Feature": fold_feature_names,
        "MeanAbsSHAP_Fold": fold_mean_abs_shap
    })

    fold_df["Rank_Fold"] = fold_df["MeanAbsSHAP_Fold"].rank(
        ascending=False,
        method="min"
    ).astype(int)

    fold_importance_tables.append(fold_df)

    if COOL_DOWN_SECONDS > 0:
        time.sleep(COOL_DOWN_SECONDS)

fold_importance_df = pd.concat(
    fold_importance_tables,
    ignore_index=True
)

fold_importance_df["Selected_TopN"] = (
    fold_importance_df["Rank_Fold"] <= SHAP_STABILITY_TOP_N
).astype(int)

importance_matrix = fold_importance_df.pivot_table(
    index="Feature",
    columns="Fold",
    values="MeanAbsSHAP_Fold",
    aggfunc="mean"
)

rank_matrix = fold_importance_df.pivot_table(
    index="Feature",
    columns="Fold",
    values="Rank_Fold",
    aggfunc="mean"
)

selection_frequency = fold_importance_df.groupby("Feature")["Selected_TopN"].mean()

fold_stability_df = pd.DataFrame({
    "Feature": importance_matrix.index,
    "Mean_SHAP_AcrossFolds": importance_matrix.mean(axis=1, skipna=True).values,
    "Std_SHAP_AcrossFolds": importance_matrix.std(axis=1, skipna=True).values,
    "Mean_Rank_AcrossFolds": rank_matrix.mean(axis=1, skipna=True).values,
    "Std_Rank_AcrossFolds": rank_matrix.std(axis=1, skipna=True).values,
    f"Selection_Frequency_Top{SHAP_STABILITY_TOP_N}": selection_frequency.reindex(
        importance_matrix.index
    ).values,
    "Available_Folds": importance_matrix.notna().sum(axis=1).values
}).sort_values(
    by=["Mean_Rank_AcrossFolds", "Mean_SHAP_AcrossFolds"],
    ascending=[True, False]
).reset_index(drop=True)

print("\n--- Fold-Level SHAP Feature-Ranking Stability ---")
print(fold_stability_df)


# -----------------------------------------------------------------------------------------
# 7) Combine final-model SHAP table with fold-level stability table
# -----------------------------------------------------------------------------------------
combined_stability_df = stability_df.merge(
    fold_stability_df,
    on="Feature",
    how="outer"
).sort_values(
    by=["Mean_Rank_AcrossFolds", "Mean_SHAP"],
    ascending=[True, False]
).reset_index(drop=True)

print("\n--- Combined SHAP Stability Table ---")
print(combined_stability_df)


# -----------------------------------------------------------------------------------------
# 8) SHAP summary plot - final fitted model
# -----------------------------------------------------------------------------------------
plt.figure(figsize=(10, 6))

shap.summary_plot(
    shap_vals_class1,
    X_shap_df,
    feature_names=feature_names,
    show=False
)

plt.title(
    f"SHAP Feature Importance - {final_model_name} "
    "(Final Fitted Model)"
)

plt.tight_layout()
plt.savefig(
    "cardio_shap_summary_stable.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()


# -----------------------------------------------------------------------------------------
# 9) SHAP dependence plot for top final-model feature
# -----------------------------------------------------------------------------------------
top_feat = stability_df.iloc[0]["Feature"]

plt.figure(figsize=(8, 6))

shap.dependence_plot(
    top_feat,
    shap_vals_class1,
    X_shap_df,
    feature_names=feature_names,
    show=False
)

plt.title(
    f"SHAP Dependence Plot: {top_feat}"
)

plt.tight_layout()
plt.savefig(
    "cardio_shap_dependence_stable.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()


# -----------------------------------------------------------------------------------------
# 10) Fold-level SHAP stability bar plot
# -----------------------------------------------------------------------------------------
top_fold_features = fold_stability_df.head(min(15, len(fold_stability_df))).copy()
top_fold_features = top_fold_features.sort_values(
    by="Mean_SHAP_AcrossFolds",
    ascending=True
)

plt.figure(figsize=(10, 7))

plt.barh(
    top_fold_features["Feature"],
    top_fold_features["Mean_SHAP_AcrossFolds"],
    xerr=top_fold_features["Std_SHAP_AcrossFolds"],
    capsize=3
)

plt.xlabel("Mean absolute SHAP across CV folds")
plt.ylabel("Feature")
plt.title(
    f"Fold-Level SHAP Feature-Ranking Stability - {final_model_name}"
)

plt.tight_layout()
plt.savefig(
    "cardio_shap_fold_stability_bar.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()


# -----------------------------------------------------------------------------------------
# 11) Save SHAP stability outputs
# -----------------------------------------------------------------------------------------
stability_df.to_csv(
    "cardio_shap_stability_values.csv",
    index=False
)

fold_importance_df.to_csv(
    "cardio_shap_fold_importance_long.csv",
    index=False
)

fold_stability_df.to_csv(
    "cardio_shap_fold_stability_values.csv",
    index=False
)

rank_matrix.to_csv(
    "cardio_shap_fold_rank_matrix.csv"
)

combined_stability_df.to_csv(
    "cardio_shap_combined_stability_values.csv",
    index=False
)

print("\n✅ SHAP Analysis completed with fold-level stability validation.")
print("• Final-model SHAP table: cardio_shap_stability_values.csv")
print("• Fold-level long table: cardio_shap_fold_importance_long.csv")
print("• Fold-level stability table: cardio_shap_fold_stability_values.csv")
print("• Fold-rank matrix: cardio_shap_fold_rank_matrix.csv")
print("• Combined stability table: cardio_shap_combined_stability_values.csv")
print("• Saved figures: cardio_shap_summary_stable.png, cardio_shap_dependence_stable.png, cardio_shap_fold_stability_bar.png")
print("• Conclusion: Feature-importance rankings were quantitatively assessed across stratified CV folds.")